# 本地 PaperQA：有答案不等于有证据支持

**问题。** 提供的文档如何变成检索片段、摘要与带引用的答案？哪些主张可以实际核对？

**学习目标。** 将原文证据与模型摘要分开检查；识别正确答案中的无依据扩写；区分拒答与
“信息不存在”的证明。需要基础 Python，以及检索增强生成的基本概念。

本 notebook 默认**读取真实运行结果**，并提供可编辑的证据检查练习。下载后可用 Jupyter，
或通过 [Colab](https://colab.research.google.com/) 的 **File → Upload notebook** 上传。
默认 cell 仅需 Python 标准库与 Jupyter 显示功能，不安装软件包、不发网络请求、不需要
API key、GPU 或模型调用。两篇附有许可和引用信息的文本及五条真实答案已嵌入文件。

最后的可选部分将本地索引和问答拆成独立可编辑 cell，需要另外准备软件和本地模型服务。
它不会自动开启，在 Colab 中也一样。

## 1. 选择模式并创建学生输出目录

离线学习时保持 `ENABLE_LOCAL_INFERENCE = False`。输出写入新目录，不覆盖作者运行得到的
答案。修改 cell 并不会更新变量，必须执行该 cell 才会生效。

`EXECUTION_PROFILE` 只被可选的本地推理部分读取。配置只设置线程数与时间上限，不改变
模型、问题、检索或生成设置；第 9 节会拒绝任何会改动这些字段的配置。阅读可选部分之前
请保持 `as-observed`，`cpu-extended` 的含义在那里说明。

In [ ]:
import asyncio
import copy
import importlib.metadata
import json
import os
import sys
import tempfile
from pathlib import Path

from IPython.display import display


def runtime_budget(profile):
    """Return transport limits only; scientific settings never depend on this."""
    budgets = {
        # Limits of the observed five-question batch, run on one GPU machine.
        "as-observed": {
            "num_thread": None,
            "chat_timeout": None,
            "embedding_timeout": None,
            "max_retries": None,
            "document_timeout": 150,
            "question_timeout": 600,
        },
        # Budget of the single accepted CPU execution copy of 10 September 2026,
        # which finished in 774.12 seconds on that machine, 708.61 of them in
        # the query. It is one machine's observation, not a validated default.
        "cpu-extended": {
            "num_thread": 4,
            "chat_timeout": 900,
            "embedding_timeout": 600,
            "max_retries": 0,
            "document_timeout": 600,
            "question_timeout": 2400,
        },
    }
    if profile not in budgets:
        raise ValueError("Unknown execution profile: " + str(profile))
    return budgets[profile]


ENABLE_LOCAL_INFERENCE = False
EXECUTION_PROFILE = "as-observed"
OUTPUT_DIR = Path(tempfile.mkdtemp(prefix="paperqa-student-", dir=Path.cwd()))
live_answers = []
print("Student output:", OUTPUT_DIR)
print("Execution profile:", EXECUTION_PROFILE, "(limits apply only to live inference)")


def save_student_execution(folder, stage, answers, events, error=None):
    """Preserve student outputs on normal stages and caught failures."""
    (folder / "student_local_answers.json").write_text(
        json.dumps(answers, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    (folder / "student_network_events.json").write_text(
        json.dumps(events, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    (folder / "student_execution.json").write_text(
        json.dumps(
            {
                "stage": stage,
                "outcome": "failed" if error is not None else "stage_finished",
                "saved_answers": len(answers),
                "error": None if error is None else f"{type(error).__name__}: {error}",
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

## 2. 载入真实示例

下方数据 cell 包含未改写的答案、检索片段、来源文章、设置与来源复核说明，不是新生成的
模型回答。可展开 cell 查看完整数据。问题的参考答案只用于教学，不会传入后面的可选模型调用。

In [ ]:
observed = json.loads('{"title": "Local-document PaperQA: observed answers and evidence", "date": "2026-09-10", "scope": "Five teaching questions about two supplied papers; not a benchmark or autonomous literature search.", "software": {"python": "3.13.13", "paperqa": "2026.8.12", "ollama": "0.33.3", "chat_model": "qwen3:8b", "embedding_model": "nomic-embed-text:v1.5"}, "observations": {"question_count": 5, "index_and_answer_seconds": 109.368, "python_peak_rss_mb": 240.3, "sampled_gpu_peak_gb": 7.334, "python_socket_events": 18, "denied_python_socket_events": 0, "observed_endpoints": ["127.0.0.1:11434", "localhost:11434"], "notes": "Runtime excludes setup/downloads. GPU memory was sampled every two seconds, not a minimum requirement or an exact peak. The Python guard does not cover native networking or the separate model server. No benchmark accuracy or human grading was measured."}, "documents": [{"id": "openbabel-2011", "pmcid": "PMC3198950", "title": "Open Babel: An open chemical toolbox", "citation": "O\'Boyle NM, Banck M, James CA, Morley C, Vandermeersch T, Hutchison GR. Open Babel: An open chemical toolbox. J Cheminform. 2011;3:33. doi:10.1186/1758-2946-3-33.", "url": "https://www.ebi.ac.uk/europepmc/webservices/rest/PMC3198950/fullTextXML", "license": "CC BY 2.0", "license_url": "https://creativecommons.org/licenses/by/2.0/", "text": "Open Babel: An open chemical toolbox\\n\\nO\'Boyle NM, Banck M, James CA, Morley C, Vandermeersch T, Hutchison GR. Open Babel: An open chemical toolbox. J Cheminform. 2011;3:33. doi:10.1186/1758-2946-3-33.\\n\\nhttps://creativecommons.org/licenses/by/2.0/\\n\\nAbstract\\n\\nBackground\\n\\nA frequent problem in computational modeling is the interconversion of chemical structures between different formats. While standard interchange formats exist (for example, Chemical Markup Language) and de facto standards have arisen (for example, SMILES format), the need to interconvert formats is a continuing problem due to the multitude of different application areas for chemistry data, differences in the data stored by different formats (0D versus 3D, for example), and competition between software along with a lack of vendor-neutral formats.\\n\\nResults\\n\\nWe discuss, for the first time, Open Babel, an open-source chemical toolbox that speaks the many languages of chemical data. Open Babel version 2.3 interconverts over 110 formats. The need to represent such a wide variety of chemical and molecular data requires a library that implements a wide range of cheminformatics algorithms, from partial charge assignment and aromaticity detection, to bond order perception and canonicalization. We detail the implementation of Open Babel, describe key advances in the 2.3 release, and outline a variety of uses both in terms of software products and scientific research, including applications far beyond simple format interconversion.\\n\\nConclusions\\n\\nOpen Babel presents a solution to the proliferation of multiple chemical file formats. In addition, it provides a variety of useful utilities from conformer searching and 2D depiction, to filtering, batch conversion, and substructure and similarity searching. For developers, it can be used as a programming library to handle chemical data in areas such as organic chemistry, drug design, materials science, and computational chemistry. It is freely available under an open-source license from http://openbabel.org.\\n\\nIntroduction\\n\\nThe history of chemical informatics has included a huge variety of textual and computer representations of molecular data. Such representations focus on specific atomic or molecular information and may not attempt to store all possible chemical data. For example, line notations like Daylight SMILES [1] do not offer coordinate information, while crystallographic or quantum mechanical formats frequently do not store chemical bonding data. Hydrogen atoms are frequently omitted from x-ray crystallography due to the difficulty in establishing coordinates, and are often ignored by some file formats as the \\"implicit valence\\" of heavy atoms that indicates their presence. Other types of representations require specification of atom types on the basis of a specific valence bond model, inclusion of computed partial charges, indication of biomolecular residues, or multiple conformations.\\n\\nWhile attempts have been made to provide a standard format for storing chemical data, including most notably the development of Chemical Markup Language (CML) [2-6], an XML dialect, such formats have not yet achieved widespread use. Consequently, a frequent problem in computational modeling is the interconversion of molecular structures between different formats, a process that involves extraction and interpretation of their chemical data and semantics.\\n\\nWe outline for the first time, the development and use of the Open Babel project, a full-featured open chemical toolbox, designed to \\"speak\\" the many different representations of chemical data. It allows anyone to search, convert, analyze, or store data from molecular modeling, chemistry, solid-state materials, biochemistry, or related areas. It provides both ready-to-use programs as well as a complete, extensible programmer\'s toolkit for developing cheminformatics software. It can handle reading, writing, and interconverting over 110 chemical file formats, supports filtering and searching molecule files using Daylight SMARTS pattern matching [7] and other methods, and provides extensible fingerprinting and molecular mechanics frameworks. We will discuss the frameworks for file format interconversion, fingerprinting, fast molecular searching, bond perception and atom typing, canonical numbering of molecular structures and fragments, molecular mechanics force fields, and the extensible interfaces provided by the software library to enable further chemistry software development.\\n\\nOpen Babel has its origin in a version of OELib released as open-source software by OpenEye Scientific under the GPL (GNU Public License). In 2001, OpenEye decided to rewrite OELib in-house as the proprietary OEChem library, so the existing code from OELib was spun out into the new Open Babel project. Since 2001, Open Babel has been developed and substantially extended as an international collaborative project using an open-source development model [8]. It has over 160,000 downloads, over 400 citations [9], is used by over 40 software projects [10], and is freely available from the Open Babel website [11].\\n\\nFeatures\\n\\nFile Format Support\\n\\nWith the release of Open Babel 2.3, Open Babel supports 111 chemical file formats in total. It can read 82 formats and write 85 formats. These encompass common formats used in cheminformatics (SMILES, InChI, MOL, MOL2), input and output files from a variety of computational chemistry packages (GAMESS, Gaussian, MOPAC), crystallographic file formats (CIF, ShelX), reaction formats (MDL RXN), file formats used by molecular dynamics and docking packages (AutoDock, Amber), formats used by 2D drawing packages (ChemDraw), 3D viewers (Chem3D, Molden) and chemical kinetics and thermodynamics (ChemKin, Thermo). Formats are implemented as \\"plugins\\" in Open Babel, which makes it easy for users to contribute new file formats (see Extensible Interface below). Depending on the format, other data is extracted by Open Babel in addition to the molecular structure; for example, vibrational frequencies are extracted from computational chemistry log files, unit cell information is extracted from CIF files, and property fields are read from SDF files.\\n\\nA number of \\"utility\\" file formats are also defined; these are not strictly speaking a way of storing the molecular structure, but rather present certain functionality through the same interface as the regular file formats. For example, the report format is a write-only utility format [12] that presents a summary of the molecular structure of a molecule; the fingerprint format [13] and fastsearch format [14] are used for similarity and substructure searching (see below); the MolPrint2D and Multilevel Neighborhoods of Atoms formats calculate circular fingerprints defined by Bender et al. [15,16] and Filimonov et al. [17,18] respectively.\\n\\nEach format can have multiple options to control either reading or writing a particular format. For example, the InChI format has 12 options including an option \\"K\\" to generate an InChIKey, \\"T <param>\\" to truncate the InChI depending on a supplied parameter and \\"w\\" to ignore certain InChI warnings. The available options are listed in the documentation, are shown in the Graphical User Interface (GUI) as checkboxes or textboxes, and can be listed at the command-line. In fact, all three are generated from the same source; a documentation string in the C++ code.\\n\\nFingerprints and Fast Searching\\n\\nDatabases are widely used to store chemical information especially in the pharmaceutical industry. A key requirement of such a database is the ability to index chemical structures so that they can be quickly retrieved given a query substructure. Open Babel provides this functionality using a path-based fingerprint. This fingerprint, referred to as FP2 in Open Babel, identifies all linear and ring substructures in the molecule of lengths 1 to 7 (excluding the 1-atom substructures C and N) and maps them onto a bit-string of length 1024 using a hash function. If a query molecule is a substructure of a target molecule, then all of the bits set in the query molecule will also be set in the target molecule. The fingerprints for two molecules can also be used to calculate structural similarity using the Tanimoto coefficient, the number of bits in common divided by the union of the bits set.\\n\\nClearly, repeated searching of the same set of molecules will involve repeated use of the same set of fingerprints. To avoid the need to recalculate the fingerprints for a particular multi-molecule file (such as an SDF file), Open Babel provides a fastindex format that solely stores a fingerprint along with an index into the original file. This index leads to a rapid increase in the speed of searching for matches to a query - datasets with several million molecules are easily searched interactively. In this way, a multi-molecule file may be used as a lightweight alternative to a chemical database system.\\n\\nBond Perception and Atom Typing\\n\\nAs mentioned above, many chemical file formats offer representations of molecular data solely as lists of atoms. For example, most quantum chemical software packages and most crystallographic file formats do not offer definitions of bonding. A similar situation occurs in the case of the Protein Data Bank (PDB) format; while standardized [19] files contain connectivity information, non-standard files exist that often do not provide full connectivity information. Consequently, Open Babel features methods to determine bond connectivity, bond order perception, aromaticity determination, and atom typing.\\n\\nBond connectivity is determined by the frequently used algorithm of detecting atoms closer than the sum of their covalent radii, with a slight tolerance (0.45 Å) to allow for longer than typical bonds. To handle disorder in crystallographic data (e.g., PDB or CIF files), atoms closer than 0.63 Å are not bonded. A further filtering pass is made to ensure standard bond valency is maintained; each element has a maximum number of bonds, if this is exceeded then the longest bonds to an atom are successively removed until the valence rule is fulfilled.\\n\\nAfter bond connectivity is determined, if needed or requested by the user, bond order perception is performed on the basis of bond angles and geometries. The method is similar to that proposed by Roger Sayle [20] and uses the average bond angle around an un-typed atom to determine sp and sp2 hybridized centers. 5-membered and 6-membered rings are checked for planarity to estimate aromaticity. Finally, atoms marked as unsaturated are checked for an unsaturated neighbor to give a double or triple bond. After this initial atom typing, known functional groups are matched, followed by aromatic rings, followed by remaining unsatisfied bonds based on a set of heuristics for short bonds, atomic electronegativity, and ring membership.\\n\\nAtom typing is performed by \\"lazy evaluation,\\" matching atoms against SMARTS patterns to determine hybridization, implicit valence, and external atom types. Atom type perception may be triggered by adding hydrogens (which requires determination of implicit and explicit valence), exporting to a file format that requires atom types, or as requested by the user. To minimize the amount of typing required, when importing from a format with atom types specified, a lookup table is used to translate between equivalent types.\\n\\nAn important part of atom typing is aromaticity detection and assignment of Kekulé bond orders (kekulization). In Open Babel, a central aromaticity model is used, largely matching the commonly used Daylight SMILES representation [1], but with added support for aromatic phosphorous and selenium. Potential aromatic atoms and bonds are flagged on the basis of membership in a ring system possibly containing 4n+2 π electrons. Aromaticity is established only if a well-defined valence bond Kekulé pattern can be determined. To do this, atoms are added to a ring system and checked against the 4n+2 π electron configuration, gradually increasing the size to establish the largest possible connected aromatic ring system. Once this ring system is determined, an exhaustive search is performed to assign single and double bonds to satisfy all valences in a Kekulé form. Since this process is exponential in complexity, the algorithm will terminate if more than 30 levels of recursion or 15 seconds are exceeded (which may occur in the case of large fused ring systems such as carbon nanotubes).\\n\\nCanonical Representation of Molecules\\n\\nIn general, for any particular molecular structure and file format, there are a large number of possible ways the structure could be stored; for example, there are N! ways of ordering the atoms in an MOL file. While each of the orderings encodes exactly the same information, it can be useful to define a canonical numbering of the atoms of a molecule and use this to derive a canonical representation of a molecule for a particular file format. For a zero-dimensional file format without coordinates, such as SMILES, the canonical representation could be used to index a database, remove duplicates or search for matches.\\n\\nOpen Babel implements a sophisticated canonicalization algorithm that can handle molecules or molecular fragments. The atom symmetry classes are the initial graph invariants and encode topological and chemical properties. A cooperative labeling procedure is used to investigate the automorphic permutations to find the canonical code. Although the algorithm is similar to the original Morgan canonical code [21], various improvements are implemented to improve performance. Most notably, the algorithm implements heuristics from the popular nauty package [22,23]. Another aspect handled by the canonical code is stereochemistry as different labelings can lead to different parities. This is further complicated by the possibility of symmetry-equivalent stereocenters and stereocenters whose configuration is interdependent. The full details will be the subject of a separate publication.\\n\\nCoordinate Generation in 2D and 3D\\n\\nOpen Babel, version 2.3, has support for 2D coordinate generation (Figure 1) through the donation of code by Sergei Trepalin, based on the code used in the MCDL chemical structure editor [24-26]. The MCDL algorithm aims to layout the molecular structure in 2D such that all bond lengths are equal and all bond angles are close to 120°. The layout algorithm includes a small database of around 150 templates to help layout cages and large fragment cycles. To deal with the problem of overlapping fragments, the algorithm includes an exhaustive search procedure that rotates around acyclic bonds by 180°.\\n\\nInterconversion of 0D, 2D and 3D structures. The structures shown are of sertraline, a selective serotonin reuptake inhibitor (SSRI) used in the treatment of depression. A SMILES string for sertraline is shown at the top; this can be considered a 0D structure (only connectivity and stereochemical information). From this, Open Babel can generate a 2D structure (bottom left, depicted by Open Babel) or a 3D structure (bottom right, depicted by Avogadro), and all of these can be interconverted.\\n\\nCoordinate generation in 3D was introduced in Open Babel version 2.2, and improved in version 2.3, to enable conversion from 0D formats such as SMILES to 3D formats such as SDF (Figure 1). The 3D structure generator builds linear components from scratch following geometrical rules based on the hybridization of the atoms. Single-conformer ring templates are used for ring systems. The template matching algorithm iterates through the templates from largest to smallest searching for matches. If a match is found, the algorithm continues but will not match any ring atoms previously templated except in the case of a single overlap (the two ring systems of a spiro group) or an overlap involving exactly two adjacent atoms (two fused ring systems). After an initial structure is generated, the stereochemistry (cis/trans and tetrahedral) is corrected to match the input structure. Finally, the energy of the structure is minimized using the MMFF94 forcefield [27-31] and a low energy conformer found using a weighted rotor search.\\n\\nWhile the 3D structure builder produces reasonable conformations for molecules without rings or with ring systems for which a template exists, the results may be poor for molecules with more complex ring systems or organometallic species. Future work will be performed to compare the results of Open Babel with other programs with respect to both speed and the quality of the generated structures [32].\\n\\nStereochemistry\\n\\nA recent focus of Open Babel development has been to ensure robust translation of stereochemical information between file formats. This is particularly important when dealing with 0D formats as these explicitly encode the perceived stereochemistry. Open Babel 2.3 includes classes to handle cis/trans double bond stereochemistry, tetrahedral stereochemistry and square-planar stereochemistry (this last is still under development), as well as perception routines for 2D and 3D geometries, and routines to query and alter the stereochemistry.\\n\\nThe detection of stereogenic units starts with an analysis of the graph symmetry of the molecule to identify the symmetry class of each atom. However, given that a complete symmetry analysis also needs to take stereochemistry into account, this means that the overall stereochemistry can only be found iteratively. At each iteration, the current atom symmetry classes are used to identify stereogenic units. For example, a tetrahedral center is identified as chiral if it has four neighbors with different symmetry classes (or three, in the case where a lone pair gives rise to the tetrahedral shape).\\n\\nForcefields\\n\\nMolecular mechanics functions are provided for use with small molecules. Typical applications include energy evaluation or minimization, alone or as part of a larger workflow. The selection of implemented force fields allows most molecular structures to be used and parameters to be assigned automatically. The MMFF94(s) force field can be used for organic or drug-like molecules [27-31]. For molecules containing any element of the periodic table or complex geometry (i.e. not supported by MMFF94), the UFF force field can be used instead [33]. Recently, code implementing the GAFF force field [34,35] was also contributed and released as part of version 2.3. All of the forcefields allow the application of constraints on particular atom positions, or particular distances.\\n\\nSeveral conformer searching methods have been implemented using the forcefields, all based on the \\"torsion-driving\\" approach. This approach involves setting torsion angles from a set of predefined allowed values for a particular rotatable bond. The most thorough search method implemented is a systematic search method, which iterates over all of the allowed torsion angles for each rotatable bond in the molecule and retains the conformer with the lowest energy. Since a systematic search may not be feasible for a molecule with multiple rotatable bonds, a number of stochastic search methods are also available: the random search method, which tries random settings for the torsion angles (from the predefined allowed values), and a weighted rotor search, a stochastic search method that converges on a low energy conformer by weighting particular torsion angles based on the relative energy of the generated conformer. With Open Babel 2.3, conformer search based on a genetic algorithm is also available which allows the application of filters (e.g. a diversity filter) and different scoring functions. This latter method can be used to generate a library of diverse conformers, or like the other methods to seek a low energy conformer [36].\\n\\nImplementation\\n\\nTechnical Details\\n\\nOpen Babel is implemented in standards-compliant C++. This ensures support for a wide variety of C++ compilers (MSVC, GCC, Intel Compiler, MinGW, Clang), operating systems (Windows, Mac OS X, Linux, BSD, Windows/Cygwin) and platforms (32-bit, 64-bit). Since version 2.3, it is compiled using the CMake build system [37,38]. This is an open-source cross-platform build system with advanced features for dependency analysis. The build system has an associated unit test framework CTest, which allows nightly builds to be compiled and tested automatically with the results collated and displayed on a centralized dashboard [39].\\n\\nTo simplify installation Open Babel has as few external dependencies as possible. Where such dependencies exist, they are optional. For example, if the XML development libraries are not available, Open Babel will still compile successfully but none of the XML formats (such as Chemical Markup Language, CML) will be available. Similarly, if the Eigen matrix and linear algebra library is not found, any classes that require fast matrix manipulation (such as OBAlign, which performs least squares alignment) will not be compiled.\\n\\nWhile the majority of the Open Babel library is written in C++, bindings have been developed for a range of other programming languages, including Java and the .NET platform, as well as the so-called \\"dynamic\\" scripting languages Perl, Python, and Ruby. These are automatically generated from the C++ header files using the SWIG tool. As described previously [40], in the case of Python an additional module is provided named Pybel that simplifies access to the C++ bindings. These interfaces facilitate development of web-enabled chemistry applications, as well as rapid development and prototyping.\\n\\nCode Architecture\\n\\nThe Open Babel codebase has a modular design as shown in Figure 2. The goal of this design is threefold:\\n\\nArchitecture of the Open Babel codebase.\\n\\n1. To separate the chemistry, the conversion process and the user interfaces reducing, as far as possible, the dependency of one upon another.\\n\\n2. To put all of the code for each chemical format in one place (usually a single file) and make the addition of new formats simple.\\n\\n3. To allow the format conversion of not just molecules, but also any other chemical objects, such as reactions.\\n\\nThe code base can be considered as consisting of the following modules (Figure 2):\\n\\n• The Chemical Core, which contains OBMol etc. and has all of the chemical structure description and manipulation. This is the heart of the application and its API can be used as a chemical toolbox. It has no input/output capabilities.\\n\\n• The Formats, which read and write to files of different types. These classes are derived from a common base class, OBFormat, which is in the Conversion Control module. They also make use of the chemical routines in the Chemical Core module. Each format file contains a global object of the format class. When the format is loaded the class constructor registers the presence of the class with OBConversion. This means that the formats are plugins - new formats can be added without changing any framework code.\\n\\n• Common Formats include OBMoleculeFormat and XMLBaseFormat from which most other formats (like Format A and Format B in the diagram) are derived. Independent formats like Format C are also possible.\\n\\n• The Conversion Control, which also keeps track of the available formats, the conversion options and the input and output streams. It can be compiled without reference to any other parts of the program. In particular, it knows nothing of the Chemical Core: mol.h is not included.\\n\\n• The User Interface, which may be a command line application, a Graphical User Interface (GUI), or may be part of another program that uses Open Babel\'s input and output facilities. This depends only on the Conversion Control module (obconversion.h is included), but not on the Chemical Core or on any of the Formats.\\n\\n• The Fingerprint API, as well as being usable in external programs, is employed by the fastsearch and fingerprint formats.\\n\\n• The Fingerprints, which are bit arrays that describe an object and which facilitate fast searching. They are also built as plugins, registering themselves with their base class OBFingerprint which is in the Fingerprint API.\\n\\n• Other features such as Forcefields, Partial Charge Models and Chemical Descriptors, although not shown in the diagram, are handled similarly to Fingerprints.\\n\\n• The Error Handling can be used throughout the program to log and display errors and warnings.\\n\\nExtensible Interface\\n\\nThe utility of software libraries such as Open Babel depends on the ability of the design to be extended over time to support new functionality. To facilitate this, Open Babel implements a plugin interface for file formats, fingerprints, charge models, descriptors, \\"operators\\" and molecular mechanics force fields. This ensures a clean separation of the implementation of a particular plugin from the core Open Babel library code, and makes it easy for a new plugin (e.g. a new file format) to be contributed; all that is needed is a single C++ file and a trivial change to one of the build files. The operator plugins provide a very general mechanism for operating on a molecule (e.g. energy minimization or 3D coordinate generation) or on a list of molecules (e.g. filtering or sorting) after reading but before writing.\\n\\nPlugins are dynamically loaded at runtime. This decreases the overall disk and memory footprint of Open Babel, allowing external developers to choose particular functionality needed for their application and ignore other, less relevant features. It also allows the possibility of a third-party distributing plugins separately to the Open Babel distribution to provide additional functionality.\\n\\nOpen-Source License and Open Development\\n\\nOpen Babel is open-source software, which offers end users and third-party developers a range of additional rights not granted by proprietary chemistry software. Open-source software, at its most basic level, grants users the rights to study how their software works, to adapt it for any purpose or otherwise modify it, and to share the software and their modifications with others. In this sense, Open Source functions in similar ways to the processes of open peer review, publication, and citation in science. The rights granted by open source licenses largely coincide with the norms of scientific ethics to enable verifiability, repeatability, and building on previous results and theories.\\n\\nBeyond these rights, Open Babel (like most other open-source projects) offers open development -- that is, all development occurs in public forums and with public code repositories. This results in greater input from the community as any user can easily submit bug reports or feature suggestions, get involved in discussions on the future direction of Open Babel or even become a developer him/herself. In practice, the number of active contributors has increased over time through this level of open, public development (Figure 3). Moreover, it means that the development of the code is completely transparent and the quality of the software is available for public scrutiny. Indeed, since its inception, over 658 bugs have been submitted to the public tracker and fixed [41].\\n\\nNumber of contributors over time. Note that this graph only includes developers who directly commited code to the Open Babel source code repository, and does not include patches provided by users.\\n\\nValidation and Testing\\n\\nOpen Babel includes an extensive test suite comprising 60 different test programs each with tens to hundreds of tests. In early 2010, a nightly build infrastructure and dashboard was put in place with support from Kitware, Inc. This has greatly improved code quality by catching regressions, and also ensures that the code compiles cleanly on all platforms and compilers supported by Open Babel. Some examples of tests that are run each night are:\\n\\n(1) The MMFF94 forcefield code is tested against the MMFF94 validation suite.\\n\\n(2) The OBAlign class, which was developed using Test-Driven Development (TDD) methodology, is run against its test suite.\\n\\n(3) Handling of symmetry is validated by converting several test cases between SMILES, 2D and 3D SDF, and InChI (there are also several test programs with unit tests for the individual stereo classes in the API).\\n\\n(4) The SMARTS parser is tested using over 250 valid and invalid SMARTS patterns, and the SMARTS matcher is tested using 125 basic SMARTS patterns.\\n\\n(5) The LSSR (Least Set of Smallest Rings) code is tested for invariance against changing the atom order for a series of polycyclic molecules.\\n\\nRecently the development team has placed a major focus on increasing the robustness of file format translation particularly in relation to the commonly used SMILES and MDL Molfile formats. Translating between these formats requires accurate stereochemistry perception, inference of implicit hydrogens, and kekulization of delocalized systems. While it is difficult to ensure that any complex piece of code is free of bugs, and Open Babel is no exception, validation procedures can be carried out to assess the current level of performance and to find additional test cases that expose bugs. The following procedure was used to guide the rewriting of stereochemistry code in Open Babel, a project that began in early 2009. Starting with a dataset of 18,084 3D structures from PubChem3D as an SDF file, we compared the result of (a) conversion to SMILES, followed by conversion of that to Canonical SMILES to (b) conversion directly to Canonical SMILES. This procedure can be used to flush out errors in reading the original SDF file, reading/writing SMILES (either due to stereochemistry errors or kekulization problems), and is also a test (to some extent) of the canonicalization code. At the time of starting this work (March 2009), the error rate found was 1424 (8%); by Oct 2009, combined work on stereochemistry, kekulization and canonicalization had reduced this to 190 (~1%), and continued improvements have reduced the number of errors down to two (shown in Figure 4) for Open Babel 2.3.1 (~0.01%). The first failure is due to a kekulization error in a polycyclic aromatic molecule incorporating heteroatoms: (a) gave c1ccc2c(c1)c1[nH][nH]c3c4c1c(c2)ccc4cc1c3cccc1 while (b) gave c1ccc2c(c1)c1nnc3c4c1c(c2)ccc4cc1c3cccc1. This error led to confusion over whether or not the aromatic nitrogens have hydrogens attached (they do not). The second failure involves confusion over the canonical stereochemistry at a bridgehead carbon: (a) gave C1CN2[C@@H](C1)CCC2 while (b) gave C1CN2[C@H](C1)CCC2. This is actually a meso compound and so both SMILES strings are correct and represent the same molecule. However the canonicalization algorithm should have chosen one stereochemistry or the other for the canonical representation.\\n\\nThe two failures found in the validation test for reading/writing SMILES.\\n\\nAnother area of focus was the canonicalization algorithm, which can be used to generate canonical SMILES as well as other formats. The algorithm can be tested by ensuring that the same canonical SMILES string is obtained even when the order of atoms in a molecule is changed (while retaining the same connection table). The test stresses all areas of the library, including aromaticity perception, kekulization, stereochemistry, and canonicalization. The development of the canonicalization code in Open Babel was guided by applying this test to the 5,151,179 molecules in the eMolecules catalogue (dated 2011-01-02) with 10 random shuffles of the atom order. At the time of the Open Babel 2.2.3 release, there were 24,404 failures of the canonicalization algorithm; this has now been reduced to only four (shown in Figure 5, < 0.001%). The Open Babel nightly test suite ensures that this test passes for a number of problematic molecules. Although the canonicalization algorithm is still not perfect, we believe that the current level of performance (99.99992% success on the eMolecules catalogue) is acceptable for general use and with time we intend to improve performance further.\\n\\nThe four failures found in the validation test for canonicalization.\\n\\nGiven that the error rate for canonicalization and handling of stereochemistry is now quite low, the next area of focus for the Open Babel development team is to improve the handling of implicit valence for \\"unusual atoms.\\" This is particularly important for organometallic species and inorganic complexes.\\n\\nUsing Open Babel\\n\\nApplications\\n\\nThe Open Babel package is composed of a set of user applications as well as a programming library. The main command line application provided is obabel (a small upgrade on the earlier babel), which facilitates file format conversion, filtering (by SMARTS, title, descriptor value, or property field), 3D or 2D structure generation, conversion of hydrogens from implicit to explicit (and vice versa), and removal of small fragments or of duplicate structures. A number of features are provided to handle multi-molecule file formats (such as SDF or MOL2) and to use or manipulate the information in property fields and molecule titles. Here is an example of using obabel to convert from SDF format to SMILES:\\n\\nobabel inputmols.sdf -O outputmols.smi\\n\\nA more complicated use would be to extract all molecules in an SDF file whose titles start with \\"active\\":\\n\\nobabel inputmols.sdf -aT -o copy -O outputmols.sdf --filter \\"title=\'active*\'\\"\\n\\nThe copy format specified by \\"-o copy\\" is a utility format that copies the exact contents of the input file (for the filtered molecules) directly to the output, without perception or interpretation. The \\"-aT\\" indicates that only the title of the input SDF file should be read; full chemical perception is not required.\\n\\nThe Open Babel graphical user interface (GUI) provides the same functionality. Figure 6 is a screenshot of the GUI carrying out the same filtering operation described in the obabel example above. The left panel deals with setting up the input file, the right panel handles the output and the central panel is for setting conversion options. Depending on whether a particular option requires a parameter, the available options are displayed either as check boxes or as text entry boxes. These interface elements are generated dynamically directly from the text description and help text provided by each format plugin.\\n\\nScreenshot of the Open Babel GUI. In the screenshot, the Open Babel GUI is running on Bio-Linux 6.0, an Ubuntu derivative.\\n\\nProgramming Library\\n\\nThe Open Babel library allows users to write chemistry applications without worrying about the low-level details of handling chemical information, such as how to read or write a particular file format, or how to use SMARTS for substructure searching. Instead, the user can focus on the scientific problem at hand, or on creating a more easy-to-use interface (e.g. a GUI) to some of Open Babel\'s functionality. The Open Babel API (Application Programming Interface) is the set of classes, methods and variables provided by Open Babel to the user for use in programs. Documentation on the complete API (generated using Doxygen [42]) is available from the Open Babel website [43], or can be generated from the source code.\\n\\nThe functionality provided by the Open Babel library is relied upon by many users and by several other software projects, with the result that introducing changes to the API would cause existing software to break. For this reason, Open Babel strives to maintain API stability over long periods of time, so that existing software will continue to work despite the release of new Open Babel versions with additional features, file formats and bug fixes. Open Babel uses a version numbering system that indicates how the API has changed with every release:\\n\\n• Bug fix releases (e.g. 2.0.0 versus 2.0.1) do not change API at all\\n\\n• Minor version releases (e.g. 2.0 versus 2.1) will add to the API, but will otherwise be backwards-compatible\\n\\n• Major version releases (e.g. 2 versus 3) are not backwards-compatible, and have changes to the API (including removal of deprecated classes and functions)\\n\\nFigure 7 shows an example C++ program that uses the two main classes OBConversion and OBMol to print out the molecular weight of all of the molecules in an SDF file. This could be used, for example, to investigate differences in the molecular weight distribution between two databases. The same program is shown in Figure 8 but implemented using the Python bindings.\\n\\nExample C++ program that uses the Open Babel library. The program prints out the molecular weight of each molecule in the SDF file \\"dataset.sdf\\".\\n\\nExample Python program that uses the Open Babel library. The program prints out the molecular weight of each molecule in the SDF file \\"dataset.sdf\\".\\n\\nExamples of Use\\n\\nOpen Babel has already been referenced over 400 times for various uses. The most common use of Open Babel is through the obabel command line application (or the corresponding graphical user interface) for the interconversion of chemical file formats. Such conversions may also involve the calculation or inference of additional molecular information or application of a filter. Some published examples of these include the following:\\n\\n• interconversion of chemical file formats or representations [44-47]\\n\\n• addition of hydrogens [48-50]\\n\\n• generation of 3D molecular structures [51-53]\\n\\n• calculation of partial charges [54,55]\\n\\n• generation of molecular fingerprints [56-59]\\n\\n• removal of duplicate molecules from a dataset [60]\\n\\n• calculation of MOL2 atom types [61]\\n\\nAn interesting example that shows how a particular chemical representation may be used to facilitate a scientific study is the crystallographic study of Fábián and Brock who used Open Babel to generate InChI strings for molecules in the Cambridge Structural Database [62]. Exploiting the fact that InChIs of enantiomers are identical expect at the enantiomer sublayer (\\"/m0\\" or \\"/m1\\"), they used the InChIs as part of a workflow to identify kryptoracemates (a class of racemic crystals where the enantiomers are not related by space-group symmetry) in the database.\\n\\nTo implement new methods, or access additional molecular information, it is necessary to use the Open Babel library directly either from C++ or using one of the supported language bindings. Some examples of published studies that have done this include the following:\\n\\n• Dehmer et al. implemented molecular complexity measures based on information theory [63].\\n\\n• Langham and Jain developed a model for chemical mutagenicity based on atom pair features [64].\\n\\n• Fontaine et al. implemented a method, anchor-GRIND, that uses an anchor point of a molecular scaffold to compare molecular interaction fields when different substituents are present [65].\\n\\n• Konyk et al. have developed a plugin for Open Babel that adds support for the Web Ontology Language (OWL) to allow automated reasoning about chemical structures [66].\\n\\n• Kogej et al. (AstraZeneca) implemented a 3-point pharmacophore fingerprint called TRUST [67].\\n\\n• Many other examples exist [68-71].\\n\\nThe vital role that a cheminformatics toolkit plays in the development of scientific resources is shown by Tables 1 and 2. Table 1 lists examples of stand-alone applications or programming libraries that rely on Open Babel, either calling the library directly or via one of the command-line executables. Table 2 contains examples of web applications and databases that either use Open Babel on the server or where Open Babel was used in the preparation of the data.\\n\\nSoftware applications and libraries that use Open Babel\\n\\nWeb applications and databases that use Open Babel\\n\\nConclusions\\n\\nIn November 2011, Open Babel will mark 10 years of existence as an independent project, and for the first time, we have discussed its development and features. As shown by more than 400 citations, it has become an essential tool for handling the myriad of molecular file formats encountered in diverse branches of chemistry. While more work remains to be done, through validation processes such as those described above and the recent introduction of a nightly build and testing framework, we aim to improve the quality and robustness of the toolkit with each new release.\\n\\nLooking forward to the future, one of the goals of the project is to extend support to molecules that currently are not handled very well by existing cheminformatics toolkits. Typically toolkits focus on the types of molecules of principal importance to the pharmaceutical industry, namely stable organic molecules comprising wholly of 2-center 2-electron covalent bonds. Molecules outside this set - such as radicals, organometallic and inorganic molecules, molecules with coordinate bonds or 3-center 2-electron bonds - are poorly supported in general. Future releases of Open Babel will provide substantially improved handling of such species. We also seek to improve speed and coverage of important methods such as structure generation, kekulization and canonicalization.\\n\\nOpen Babel is freely available from http://openbabel.org, and new community members are very welcome (users, developers, bug reporters, feature requesters). For information on how to use Open Babel, please see the documentation at http://openbabel.org/docs and the API documentation at http://openbabel.org/api.\\n\\nAvailability and Requirements\\n\\nProject Name: Open Babel\\n\\nProject home page: http://openbabel.org\\n\\nOperating system(s): Cross-platform\\n\\nProgramming language: C++, bindings to Python, Perl, Ruby, Java, C#\\n\\nOther requirements (if compiling): CMake 2.4+\\n\\nLicense: GNU GPL v2\\n\\nAny restrictions to use by non-academics: None\\n\\nCompeting interests\\n\\nThe authors declare that they have no competing interests.\\n\\nAuthors\' contributions\\n\\nGRH is the lead developer of the Open Babel project. CAJ, CM, MB, NMOB, and TV are developers of Open Babel. All authors read and approved the final manuscript.\\n\\nContributor Information\\n\\nNoel M O\'Boyle, Email: baoilleach@gmail.com.\\n\\nMichael Banck, Email: michael.banck@ch.tum.de.\\n\\nCraig A James, Email: cjames@emolecules.com.\\n\\nChris Morley, Email: c.morley@gaseq.co.uk.\\n\\nTim Vandermeersch, Email: tim.vandermeersch@gmail.com.\\n\\nGeoffrey R Hutchison, Email: geoffh@pitt.edu.\\n\\nAcknowledgements and Funding\\n\\nWe would like to thank all users and contributors to the Open Babel project over its history, including OpenEye Scientific Software Inc. for their initial OELib code. We also thank the Blue Obelisk Movement for ideas, comments on this manuscript, and support. We thank SourceForge for providing resources for issue tracking and managing releases, and Kitware for additional dashboard resources. NMOB is supported by a Health Research Board Career Development Fellowship (PD/2009/13).\\n\\nReferences\\n"}, {"id": "cmms-gcl-2023", "pmcid": "PMC10457661", "title": "CMMS-GCL: cross-modality metabolic stability prediction with graph contrastive learning", "citation": "Du B-X, Long Y, Li X, Wu M, Shi J-Y. CMMS-GCL: cross-modality metabolic stability prediction with graph contrastive learning. Bioinformatics. 2023;39(8):btad503. doi:10.1093/bioinformatics/btad503.", "url": "https://www.ebi.ac.uk/europepmc/webservices/rest/PMC10457661/fullTextXML", "license": "CC BY 4.0", "license_url": "https://creativecommons.org/licenses/by/4.0/", "text": "CMMS-GCL: cross-modality metabolic stability prediction with graph contrastive learning\\n\\nDu B-X, Long Y, Li X, Wu M, Shi J-Y. CMMS-GCL: cross-modality metabolic stability prediction with graph contrastive learning. Bioinformatics. 2023;39(8):btad503. doi:10.1093/bioinformatics/btad503.\\n\\nhttps://creativecommons.org/licenses/by/4.0/\\n\\nAbstract\\n\\nMotivation\\n\\nMetabolic stability plays a crucial role in the early stages of drug discovery and development. Accurately modeling and predicting molecular metabolic stability has great potential for the efficient screening of drug candidates as well as the optimization of lead compounds. Considering wet-lab experiment is time-consuming, laborious, and expensive, in silico prediction of metabolic stability is an alternative choice. However, few computational methods have been developed to address this task. In addition, it remains a significant challenge to explain key functional groups determining metabolic stability.\\n\\nResults\\n\\nTo address these issues, we develop a novel cross-modality graph contrastive learning model named CMMS-GCL for predicting the metabolic stability of drug candidates. In our framework, we design deep learning methods to extract features for molecules from two modality data, i.e. SMILES sequence and molecule graph. In particular, for the sequence data, we design a multihead attention BiGRU-based encoder to preserve the context of symbols to learn sequence representations of molecules. For the graph data, we propose a graph contrastive learning-based encoder to learn structure representations by effectively capturing the consistencies between local and global structures. We further exploit fully connected neural networks to combine the sequence and structure representations for model training. Extensive experimental results on two datasets demonstrate that our CMMS-GCL consistently outperforms seven state-of-the-art methods. Furthermore, a collection of case studies on sequence data and statistical analyses of the graph structure module strengthens the validation of the interpretability of crucial functional groups recognized by CMMS-GCL. Overall, CMMS-GCL can serve as an effective and interpretable tool for predicting metabolic stability, identifying critical functional groups, and thus facilitating the drug discovery process and lead compound optimization.\\n\\nAvailability and implementation\\n\\nThe code and data underlying this article are freely available at https://github.com/dubingxue/CMMS-GCL.\\n\\n1 Introduction\\n\\nMetabolic stability, which refers to the extent and rate of drug metabolism in the body, plays a crucial role in the early stages of drug discovery and development (Pritchard et al. 2003). It significantly impacts various pharmacokinetic (PK) processes, such as oral bioavailability, the volume of distribution, clearance, half-life, and toxicity, ultimately influencing the appropriate drug dose and frequency (Davies et al. 2020, Gajula et al. 2021). Therefore, improving the metabolic stability of hit/lead compounds in the early phases of drug discovery has a pivotal role w.r.t. drug candidate screening and lead compound optimization (Kirchmair et al. 2015).\\n\\nMost of the early metabolic stability assays are conducted in vitro by incubating hit/lead compounds with liver microsomes (LMs) to assess their metabolic properties (Gajula et al. 2021). Notably, LMs are abundant in subcellular components containing cytochrome P450 (CYP450) enzymes, which play a crucial role in drug metabolism, and their assays provide preliminary guidance for more definitive PK assessments (Słoczyńska et al. 2019). Furthermore, in vitro mouse liver microsomes (MLMs), rat liver microsomes (RLMs), and human liver microsomes (HLMs) have emerged as essential approaches for initial assessment of metabolic stability (Mak et al. 2022, Rodríguez-Pérez et al. 2023). However, experimental determination of molecular metabolic stability is time-consuming, laborious, and expensive (Sodhi and Benet 2021). Therefore, there is an urgent need to develop computational methods to assist experimental screenings of metabolic stability.\\n\\nIn the past several years, traditional machine learning-based methods have been developed for metabolic stability prediction. For example, Perryman et al. (2016) first collected a dataset of MLM compounds from PubChem and presented a Bayesian model to predict molecular metabolic stability using nine molecular descriptors (e.g. fingerprints). Podlewska and Kafel (2018) built an online platform called MetStabOn for metabolic stability prediction, where multiple kinds of traditional machine learning methods were assembled, such as k-nearest neighbor (kNN), Naïve Bayes (NB), and random forest (RF). MetStabOn takes PaDEL-descriptors and extended fingerprints as input and can be applied for MLMs, RLMs, and HLMs data. Recently, Ryu et al. (2022) first generated an in-house HLMs dataset and introduced a RF model to evaluate metabolic stability with molecular descriptors as inputs. However, these methods take fingerprint or descriptor information into account only, while ignoring the molecule structural information, which results in suboptimal performance.\\n\\nThe rapid development of machine learning has led to the widespread application of graph neural networks (GNNs) in various biological research fields, such as drug–target interaction prediction (Long et al. 2022, Wang et al. 2022), molecular property prediction (Wu et al. 2018, Chen et al. 2021), and ADMET prediction (Feinberg et al. 2020, Xiong et al. 2021, Du et al. 2023). The compound molecule can be characterized as a graph structure, and motivated by this fact, several studies have recently attempted to use GNNs to predict metabolic stability. For example, Renn et al. (2021) first converted compound SMILES sequences to graph structures and then implemented graph convolutional networks on the graph structures for metabolic stability prediction. This method learned molecule representations by capturing both local and global chemical properties. Very recently, Rodríguez-Pérez et al. (2023) proposed a GNN-based assemble method called MT-GNN to evaluate metabolic stability, which can make predictions across multiple species. Besides, Li et al. (2022) developed a directed message-passing neural network (D-MPNN) for metabolic stability prediction based on the molecular graph.\\n\\nNevertheless, existing methods have two main limitations. First, most of them focus solely on using either SMILES sequence or molecular structure for metabolic stability prediction. While SMILES sequences provide information about the context of symbols and molecular structure determines the physical and chemical properties of molecules, combining both modalities allows combining SMILES sequence with molecular structure allows deep learning methods for learning more informative features than single modality sequence or structure data. Second, identifying the key functional groups that determine metabolic stability is crucial for lead compound optimization and drug design. Unfortunately, only a few previous methods have been developed to deal with this task.\\n\\nTo address these challenges, we propose a cross-modality graph contrastive learning model named CMMS-GCL for metabolic stability prediction. In the model, we design a dual representation of learning channels for molecules. Specifically, with SMILES sequence data as inputs, we first design a multihead Bi-directional Gated Recurrent Unit (BiGRU)-based encoder to learn the sequence representation of molecules by fully preserving the context of symbols. To capture the consistencies between local and global structures, we propose a graph contrastive learning framework to learn different structure representations. To the best of our knowledge, CMMS-GCL is the first method to simultaneously consider SMILES sequence and molecule structure for metabolic stability prediction. Comprehensive experiments show that our CMMS-GCL outperforms state-of-the-art methods. In addition, case studies further validate the interpretability of critical functional groups identified by CMMS-GCL. Overall, the main contributions can be summarized as follows:\\n\\nWe propose a novel graph contrastive learning model named CMMS-GCL for metabolic stability prediction. Specifically, CMMS-GCL is an interpretable model, adept at identifying key functional groups crucial for metabolic stability. The model achieves this by analyzing specific sequence cases, with further validation provided by statistical analyses rooted in the graph structure.\\n\\nWe design a new cross-modality representation learning framework where SMILES sequence and molecule structures are considered simultaneously to learn molecule features. To the best of our knowledge, this is the first attempt to integrate both modality data for effective metabolic stability prediction.\\n\\nComprehensive experiments are conducted on two datasets and the results show that our CMMS-GCL outperforms 7 state-of-the-art methods. Case studies on several stable/unstable molecules further validate the interpretability of key functional groups identified by CMMS-GCL.\\n\\n2 Methodology\\n\\n2.1 Problem formulation\\n\\nWe model the metabolic stability prediction problem as a classification task. Assuming that we are given a set of n compounds D={Di|i∈{1,…,n}} and a set of metabolic stability labels y={yi|yi∈{0,1},i∈{1,…,n}} for each compound Di, where yi=1 represents the i-th compound is stable; otherwise unstable. Our main task is to train a classifier fθ to assign probability scores y^ to the compounds to determine whether these compounds are stable. Note that θ denotes the classifier parameters.\\n\\n2.2 Overview of CMMS-GCL\\n\\nIn this article, we propose a novel cross-modality graph contrastive learning framework called CMMS-GCL for metabolic stability prediction. As shown in Fig. 1, CMMS-GCL mainly consists of an atomic similarity-based sequence encoder, a molecular graph structure encoder, an inter-view graph contrastive learning and a stability predictor. In particular, the atomic similarity-based sequence encoder captures the atomic semantically similar mapping, the molecular graph structure encoder represents the global structure of atoms and bonds, the interview graph contrastive learning module generates local structure and preserves consistencies with global structure, and the stability predictor estimates the stability of a given molecular structure based on the learned representations. Next, we will introduce each module in detail.\\n\\nThe overall framework of CMMS-GCL for metabolic stability prediction. CMMS-GCL is a cross-modality fusion framework augmented via graph contrastive learning, which contains an atomic similarity-based sequence encoder, a molecular graph structure encoder, an inter-view graph contrastive learning, and a stability predictor. Specifically, the atomic similarity-based sequence encoder uses a multihead BiGRU to generate molecular sequence embeddings, where the initial features are obtained by a pretrained Sim2Vec. The molecule graph structure encoder generates molecule embeddings by two-layer GINs. The inter-view graph contrastive learning module augments the original molecular graph by substructure sampling and then contrasts the augmented molecular graph embeddings and the original molecular graph embeddings. Finally, the sequence embeddings and graph embeddings are concatenated and fed into the final predictor for metabolic stability prediction.\\n\\n2.3 Atomic similarity-based sequence encoder\\n\\nCompounds are usually represented by the SMILES (Simplified Molecular-Input Line-Entry System). The local chemical context of an atom reveals its functionality in the SMILES sequence. To capture the context information, we first learn the preliminary features of atoms from SMILES sequence data using Smi2Vec (Lin et al. 2020a,b), which is shown to be capable of preserving the proximity of semantically similar mappings in the embedding space. Following that, we further design an atomic similarity-based sequence encoder to learn representations of compounds. Specifically, the Smi2Vec algorithm first divides a given SMILES string into individual atom or symbol and then looks up the embedding for each atom from a pretrained embedding dictionary. If an atom is not in the dictionary, a random embedding is generated. With the output of Smi2Vec, we can derive the pretrained atom representations Z∈Rm×d1, where each row represents the pretrained representation of an atom. m and d1 denote the number of atoms and the dimension of representations, respectively.\\n\\nAfter deriving the preliminary atom representations, we further design a BiGRU-based encoder to learn representations for compounds. GRU aims to enforce every recurrent unit to preserve the dependencies of different time scales adaptively. Therefore, as a variant of GRU, BiGRU allows the encoder to capture the local chemical context of atoms. Mathematically, the hidden state ht of the t-th atom in the sequence can be formulated as follows: where GRU→(⋅) and GRU→(⋅) represent the forward and backward GRUs, respectively, which captures the interdependence between physically adjacent atoms within the sequence. zt is the preliminary representation vector of the t-th atom in the sequence. zt−1→ and zt+1→ denote the hidden states of the (t−1)-th and (t+1)-th atoms from the forward and backward GRUs, respectively. As the output of the BiGRU model, we can generate the updated atom representation H∈Rm×d2 with dimension d2.\\n\\nPrevious study shows that key functional groups (i.e. molecule substructure) play a critical role in metabolic stability (Li et al. 2022). To effectively identify key functional groups, we introduce an attention mechanism (Zheng et al. 2020) to the BiGRU model. In particular, given a molecule i, we first learn the importance of all atoms by the following attention coefficient: where αij∈R1×m is attention coefficient that denotes the importance of atom j to the molecule i. MLP(⋅) is a trainable feedforward neural network with an activation function (i.e. tanh). Softmax[⋅] is a normalization function that ensures the sum of attention scores of all atoms is equal to 1. Subsequently, we derive the molecule representation si by aggregating the representation of all the atoms according to their attention coefficients: where FCs(⋅) is a fully connected neural network.\\n\\n2.4 Molecular graph structure encoder\\n\\nIn this section, we introduce the graph isomorphism network (GIN) to learn the representation of molecules from molecular structures. Given a molecule i, let us denote its graph structure as G={V,E}, where V is a set of N nodes (i.e. atoms) and E is a set of edges (i.e. chemical bonds connecting atoms). We denote A∈RN×N as the adjacency matrix of the graph, in which avu=1 indicates atoms v and u are connected by a bond while avu=0 indicates atoms v and u are disconnected by a bond. N is the number of atoms. The initial input features of the molecule i are defined by several atom-level properties such as the atom symbol, the number of adjacent atoms, the number of adjacent hydrogens, the implicit value of the atom, and the atom occurrence in an aromatic structure.\\n\\nFor each atom v in the molecular graph G, GIN learns its representation by iteratively aggregating the representations of its neighbors. Specifically, the representation hv(k) for atom v in the kth layer is defined as follows: where ϵ(k) is a learnable weight parameter. hv(k) is the representation of node v at the k-th iteration. MLP(⋅) is a feedforward neural network. hv(k−1) is the representation of node v at the (k−1)-th iteration. N(v) denotes the set of neighbors of the node v. With K iterations, the representation hv can effectively capture K-hop neighborhood information. Eventually, we take the output of the K-th iteration as the final representation hv for the atom v. To obtain the molecule representation pi∈R1×d3 with dimension d3, we first implement global max-pooling and global mean-pooling on the atom representation hv, respectively, and then concatenate two pooled representation vectors.\\n\\n2.5 Inter-view graph contrastive learning\\n\\nInspired by the study that preserving consistencies between local and global structures in a graph is demonstrated to improve graph representation learning (You et al. 2020), we further introduce graph contrastive learning to enhance the representation learning for molecules.\\n\\nGiven a molecular graph G={V,E}, we adopt substructure sampling-based augmentation to generate a local subgraph, denoted as G^={V^,E^}, where V^ and E^ are the sets of nodes and edges respectively and |V^|≤k|V|. Suppose we randomly sample a mini-batch of Q molecular graphs and, through data augmentation, derive an equal number of corresponding subgraphs to enhance the learning process. In particular, for a given molecule i, its graph representation and its corresponding graph representation from the augmented subgraph form a positive pair, while its representation and the representations of the remaining 2(Q-1) graphs/subgraphs form negative pairs. The primary idea of graph contrastive learning is to maximize the agreement between the representations of positive pairs while minimizing the agreement between the representations of negative pairs (Chen et al. 2020). With a given molecule graph i and its corresponding subgraph as inputs, we can attain its representations zi and its subgraph representation zi^ via GIN-based encoder. The contrastive loss is defined as follows: where sim(zi,zi^)=zi⊤zi^/‖uzi||||zi^|| denotes the similarity of the representations zi and zi^, i.e. cosine similarity. τ is the temperature parameter (set to 0.2 by default). As such, for the input batch samples, their overall contrastive loss is defined as follows:\\n\\n2.6 Stability predictor\\n\\nFor a given molecule i, after deriving its sequence representation si and structure representation zi, we first concatenate them and then feed them into a classifier. where y^i is a probability score that represents the possibility of being stable or unstable. The classification loss function is defined as follows: where yi∈{0,1} is the true label (i.e. stable or unstable), and Q is the number of molecules in the mini-batch samples. Subsequently, the overall loss function L is defined in Equation (10). where λ is the weight coefficient that is used to trade-off the contributions of classification loss and contrastive learning loss.\\n\\n3 Experiments\\n\\nIn this section, we first introduce the experimental setups and then validate the performance of our proposed CMMS-GCL by comparing it with baselines, conducting an ablation study, and presenting a case study.\\n\\n3.1 Experimental setups\\n\\n3.1.1 Datasets and evaluation metrics\\n\\nThe first metabolic stability dataset used in our experiments denoted as HLM dataset, was obtained from literature (Li et al. 2022). The authors collected human liver microsomal data from the ChEMBL bioactivity database. As defined in an existing study (Shah et al. 2020), molecules with a half-life of <30 min or 50% remaining after 30 min are considered stable; otherwise unstable. Hence, in this work, we determined the metabolic stability of molecules with a threshold of half-life within 30 min. We removed duplicate molecules and finally obtained 5878 molecules for HLM dataset, including 3784 stable and 2094 unstable compounds. In addition, following (Li et al. 2022), we evaluated the performance of our CMMS-GCL model on an external dataset consisting of a total of 111 molecules, of which 82 are stable and 29 are unstable. Table 1 summarizes the details of these two datasets.\\n\\nStatistics of metabolic stability datesets.\\n\\nFurthermore, we assessed the molecular structural similarity between the molecules in the External and HLM datasets. We used the Extended Connectivity Fingerprints (ECFP) for molecules and calculated their Tanimoto similarity, a robust measure of structural similarity between molecules (Bajusz et al. 2015, Ucak et al. 2021). Note that the range of the Tanimoto similarity is between 0 and 1, where 0 indicates entirely dissimilar structures and 1 means identical molecules. Given a molecule in the external dataset, we calculated its Tanimoto similarity scores to all the molecules in the HLM dataset, and considered the maximum similarity score as the similarity between this given molecule and the HLM dataset (Fig. 2). Following the previous study (He et al. 2022), compounds are generally classified into three categories based on their Tanimoto similarity scores: up to 0.5 (dissimilar), between 0.5 and 0.7 (moderately similar), and greater than 0.7 (highly similar). Within these similarity scores in our dataset, we found 70.27% were dissimilar, 23.42% were moderately similar, and a minor 6.31% were highly similar, resulting in an overall mean Tanimoto similarity score of 0.47, thus underscoring the low degree of similarity between our external and HLM datasets.\\n\\nDistribution of the maximum Tanimoto similarity scores of each molecule in the external dataset when compared with the entire HLM dataset.\\n\\nWe performed two types of evaluation experiments on the above two datasets. In particular, we implemented 10-fold cross-validation on the HLM dataset for performance evaluation. And we repeat the experiment 10 times and take average values as the final result on the external dataset. We further conducted the external independent evaluation, by training the model on the HLM dataset and testing on the external dataset. Three well-known performance metrics are utilized for evaluation, including the average area under the receiver operating characteristic curve (AUC), Accuracy, F1 score, and Matthew’s correlation coefficient (MCC). For all these three metrics, a higher value means better performance.\\n\\n3.1.2 Implementation details\\n\\nIn the atomic similarity-based sequence encoder, the dimensions d1 and d2 of the pretrained and output atom representations are set to 100 and 200, respectively. In the molecular graph structure encoder, we initially represented each node of the input molecule graph using a binary atom feature vector with 84 dimensions. We set the number K of GIN layers as 2. In the stability predictor, the dimensions of the input layer, hidden layers, and output layer are set to 768, 256, and 1, respectively.\\n\\nTo train the model of CMMS-GCL, we set training epochs and learning rate as 200 and 0.0005, respectively. Adam (Kingma and Ba. 2015) is selected as the optimizer. For data augmentation, the sampling ratio k is set as 0.4. We evaluate the influence of the weight coefficient λ by testing values in {0.01,0.05,0.1,0.25,0.5,0.75,1,1.5,2} and find that the best performance is achieved with λ = 0.1.\\n\\nFurthermore, CMMS-GCL is implemented in Python 3.8 and PyTorch 2.0, along with functions from PyG 2.3.1, Networkx 2.8.4, Scikit-learn 1.2.2, Numpy 1.23.5, Pandas 2.0.2, and RDKit 2023.03.1.\\n\\n3.2 Comparisons with baselines\\n\\nTo evaluate the performance of the proposed method, we compare CMMS-GCL with seven state-of-the-art methods, including three traditional machine learning-based methods and four deep learning-based methods, summarized as follows:\\n\\nFP-GBDT (Li et al. 2022): we implemented GBDT in the exact same way as that in (Li et al. 2022), which used ECFP fingerprints as input with the optimized parameters setting where the maximum depth of the tree is 4, the number of decision trees is 1200 and the minimum number of samples required to be at a leaf node was set to 9.\\n\\nFP-XGBoost (Li et al. 2022): the implementation of XGBoost is similar to FP-GBDT, which also used ECFP fingerprints as input. Optimized parameter settings adopted by the references include a maximum tree depth of 3, an ensemble of 900 decision trees, and a minimum requirement of 5 samples per leaf node.\\n\\nPredMS (Ryu et al. 2022): it used highly important molecular descriptors features as input and then sent into RF-based feature selection with the parameter setting where the maximum depth of the tree is 30, the number of decision trees is 500 and the minimum number of samples required to be at a leaf node was set to 3.\\n\\nGCN (Renn et al. 2021): it implemented the same molecule graph features of CMMS-GCL and then used a two-layer GCN to a feed-forward neural network. Its default values of parameters were used to train the model in the following experiments.\\n\\nD-MPNN (Li et al. 2022): it adopted 133 features for each atom and 14 features for each bond in a molecule and it performed 6 message-passing iterations on the molecular graph and then used 2 feed-forward layers.\\n\\nGAT (Veličković et al. 2018): it adopted the same molecular graph features of CMMS-GCL and then used two-layer GAT with 2 heads to a feed-forward neural network. The parameters in the feed-forward neural network are the same as those in CMMS-GCL.\\n\\nAttentiveFP (Xiong et al. 2020): it implemented the same node features of CMMS-GCL and extra 10-dim bond features as input. Then it used a three-layer message-passing neural network and two-time steps in the message-passing process.\\n\\nThe first five approaches utilized the parameter settings recommended by the original reference literature, while for the latter two methods, we identified the optimal settings.\\n\\n3.3 Performance evaluation on HLM dataset\\n\\nTable 2 shows the results of 10-fold cross-validation on HLM dataset. Among all the methods, CMMS-GCL achieves the best performance with an average AUC of 0.866±0.013, average Accuracy of 0.805±0.016, average F1 score of 0.853±0.011, and average MCC of 0.570±0.034, which are 1.05%, 1.00%, 0.59%, and 1.42% higher than that of the second-best methods. The results demonstrate the effectiveness of CMMS-GCL in predicting the metabolic stability of compound candidates.\\n\\nPerformance evaluation of CMMS-GCL with baseline methods on HLM metabolic stability prediction.\\n\\nThe best results are marked in bold and the second best is underlined.\\n\\nThere are two main reasons why our model outperforms baselines. On the one hand, CMMS-GCL combines both compound SMILES sequences with molecular structures, which allows our model to learn more informative representations than baseline methods that use single modality data (i.e. Ryu et al. 2022). On the other hand, in the molecular graph structure encoder, we introduce graph contrastive learning to capture the dependencies between local and global structures, which enhances the representation learning of our model. We will demonstrate it in the following ablation study.\\n\\n3.4 Independent evaluation on the external dataset\\n\\nTo further validate the effectiveness and generalizability of our model, we carried out CMMS-GCL and baseline methods on an external dataset. Note that all the methods utilize the HLM dataset for model training and use the external data as a test dataset.\\n\\nThe results in Table 3 show that CMMS-GCL achieved the best performance compared with seven methods across all metrics on 10 times average results. More specifically, CMMS-GCL surpasses the second-best method by 6.60%, 8.17%, 4.45%, and 27.31% in terms of AUC, Accuracy, F1 score, and MCC, respectively. Furthermore, we derived a subset of molecules from the external dataset, which shows low similarity to the HLM dataset. We also compared our CMMS-GCL with various baselines on this subset of molecules with low structural similarity (<0.5) as shown in Table 4. We can observe the improvement of our method upon the second-best by 7.72%, 6.31%, 3.19%, and 38.60% in terms of AUC, Accuracy, F1 score, and MCC, respectively. More importantly, compared with the results on the HLM dataset (Table 2), CMMS-GCL exhibited a more substantial improvement on the external dataset (Tables 3 and 4). In contrast, for baseline methods, especially methods that are based on traditional machine learning, their performance decrease to varying degrees. Therefore, we can make a conclusion that the proposed CMMS-GCL is an effective and robust computational model for predicting the metabolic stability of compounds.\\n\\nPerformance evaluation of CMMS-GCL with baseline methods on the external datasets.\\n\\nThe best results are marked in bold and the second best is underlined.\\n\\nPerformance evaluation of CMMS-GCL against baseline methods using a subset of external datasets with low molecular similarity.\\n\\nThe best results are marked in bold and the second best is underlined.\\n\\n3.5 Ablation studies\\n\\nOur CMMS-GCL consists of an atom similarity-based sequence encoder, a molecular graph structure encoder, an inter-view graph contrastive learning, and a stability predictor, as shown in Fig. 1. In this section, we perform an ablation study on two datasets to evaluate the contributions of the cross-modality encoder and the inter-view graph contrastive learning. Note that in Tables 5 and 6, “w/o CM” denotes the variant of CMMS-GCL only reserves molecular graph structure encoder and stability predictor. “w/o GCL” denotes the variant of CMMS-GCL without inter-view graph contrastive learning. Tables 5 and 6 report the performance comparison between CMMS-GCL and its variants on HLM and the external dataset in terms of four metrics, respectively. It can be discovered that both cross-modality encoder and inter-view graph contrastive learning contribute to the performance improvement of the model. Specifically, in Table 5, compared with the variants w/o CM and w/o GCL, CMMS-GCL improves on AUC by 0.81% and 0.58%, on Accuracy by 1.13% and 0.75%, on F1 score by 1.55% and 0.95%, on MCC by 1.60% and 0.88%, respectively. Moreover, on the external dataset, compared with the variants w/o CM and w/o GCL, CMMS-GCL improves on AUC by 2.19% and 1.49%, on Accuracy by 2.46% and 1.46%, on F1 score by 1.48% and 0.56%, on MCC by 14.40% and 12.13%, respectively.\\n\\nAblation studies of CMMS-GCL, which include two components variants and three augmentation methods variants.\\n\\nThe best results are marked in bold.\\n\\nAblation studies of CMMS-GCL on external dataset, which include two components variants and three augmentation methods variants.\\n\\nThe best results are marked in bold.\\n\\nMoreover, to further validate the superiority of the substructure sampling-based graph augmentation method, we compared it with other three augmentation methods: Node Dropping, Edge Perturbation, and Attribute Masking. The sampling ratios, which determine the fraction of nodes/edges/features selected by these three augmentations, were set to 0.2, 0.4, and 0.4, respectively. The results in Tables 5 and 6 show that the substructure sampling-based method achieves the best performance consistently and improves the second-best method by 0.35%, 0.37%, 0.35%, and 1.24% in terms of AUC, accuracy, F1 score, and MCC, respectively, in Table 5 and improves the second-best method by 1.60%, 3.09%, 1.60%, and 20.51% in terms of AUC, accuracy, F1 score, and MCC, respectively, in Table 6. The results indicate that the substructure sampling-based graph contrastive learning plays a vital role in predicting metabolic stability.\\n\\n3.6 Performance evaluation on novel structural molecules\\n\\nTo validate our CMMS-GCL’s capacity for novel molecular structures, we calculated the Tanimoto similarity among the molecules within the HLM dataset using their ECFP fingerprints. We then performed a K-means clustering analysis on the similarity matrix, resulting in five clear clusters. To present our results more tangibly, we employed Principal Component Analysis (PCA) for dimensionality reduction, as depicted in Fig. 3. This dimension reduction allowed us to visualize and interpret the clustering results more effectively, thereby accentuating the efficiency of our CMMS-GCL in identifying distinct molecular structures.\\n\\nPCA visualization of K-means clustering on the HLM dataset, where the values in parentheses correspond to the explained variance ratio.\\n\\nFurthermore, we conducted the leave-one-cluster-out cross-validation on the identified clusters, providing a robust validation of our approach on the molecules with novel structures. We compared our CMMS-GCL against various baseline methods under this leave-one-cluster-out setting in Fig. 4. We can observe that CMMS-GCL outperforms the second-best method by 1.99%, 0.24%, 0.36%, and 3.82% in terms of AUC, Accuracy, F1 score, and MCC, respectively.\\n\\nPerformance comparison of CMMS-GCL and baseline methods on HLM dataset based on leave-one-cluster-out strategy.\\n\\nIn conclusion, the assessment of structurally unique compounds manifests the superior predictive performance of CMMS-GCL, underscoring its competency in handling diverse chemical structures. This revelation accentuates the potential of CMMS-GCL as an invaluable asset for medicinal chemists in pinpointing compounds with novel structures and predicting their respective activities.\\n\\n3.7 Case study on interpretability of CMMS-GCL\\n\\nAlthough deep learning is often known as a black-box model, it is crucial to comprehend the prediction mechanism employed by CMMS-GCL for distinguishing stable and unstable molecules, as well as to ascertain its potential influence on lead compound optimization. To thoroughly investigate the interpretability of our model, we embarked on an in-depth examination of its two primary modules, focusing distinctly on the sequence-based and the graph structure-based components.\\n\\n3.7.1 Sequence-based interpretability\\n\\nSince the atomic similarity-based sequence encoder module can capture the molecule’s important functional groups through its multihead attention module, we determined that two bonding atoms are regarded as a crucial functional group. The weight of a bond is the average of the weights of its constituent atoms (partial bonds that can be visualized are also averaged together of SMILES symbols) and are highlighted in Fig. 5.\\n\\nInterpretability case study of crucial functional groups for stable and unstable molecules based on sequences. (a) Four stable molecules with high metabolic stability. (b) Four unstable molecules with low metabolic stability. The atoms and bonds of high/low metabolic stability-specific functional groups are highlighted.\\n\\nWe selected four molecules exhibiting high metabolic stability. The highlighted regions reveal specific areas within the molecules that contribute to metabolic stability or instability. As shown in Fig. 5a, for the first two stable molecules, their highlighted regions exhibit acetylene functional groups, which are commonly associated with enhanced stability and are less prone to metabolic reactions (Talele 2020). For the other two molecules, the trifluoromethyl group exhibits strong electronegativity and chemical inertness, making it difficult to undergo a transformation during metabolic processes in biological systems (Liang et al. 2013).\\n\\nIn contrast, another four molecules display low metabolic stability (i.e. instability). For these unstable molecules prone to metabolism, their prominent functional groups involve various metabolic pathways. As shown in Fig. 5b, the attention scores indicate that specific functional groups, such as amide, thiophene, hydroxyl, ester, methoxy, and nitrogen-containing heterocycles, receive the highest scores, making them crucial for the model’s prediction. Furthermore, amide more readily undergo hydrolysis (Casey Laizure et al. 2013), and thiophene rings can be oxidized by CYP450 enzymes, leading to the formation of S-oxides (Gramec et al. 2014). Also, hydroxyl groups are susceptible to oxidative reactions catalyzed by CYP450 enzymes, such as hydroxylation, resulting in the formation of quinone or quinone-type phenolic derivatives (Zhang and Tang 2018). Ester functional groups are typically susceptible to hydrolysis by esterases in biological systems (Abelian et al. 2021), resulting in the formation of corresponding alcohols and acids. Lastly, the methoxy group can potentially be metabolized by enzymes such as cytochrome P450 or O-demethylases, converting it to the relevant alcohol and nitrogen-containing heterocycles tend to undergo oxidation reactions, forming N-oxides on the nitrogen atom, increasing the compound’s polarity, and facilitating its excretion in biological systems(Sharma et al. 2018, Makhova et al. 2020).\\n\\n3.7.2 Graph structure-based interpretability\\n\\nIn terms of the graph structures, we employed the EdgeSHAPer method, which calculates the Shapley value for each edge as if they were individual contributors to the model’s predictive capability (Mastropietro et al. 2022). The Shapley value here represents each edge’s (or bond’s) average marginal contribution to the model’s collective prediction of metabolic stability and instability. In order to attain more comprehensive statistics related to the chemical groups that influence metabolic stability, we utilized the trained model on the first fold of the HLM dataset and executed interpretability analysis on the corresponding test set, comprising 383 positive and 202 negative samples. Through a statistical analysis focused on these critical bonds and their neighbour bonds, we ultimately discerned some general statistical patterns. The statistical results of stability and instability are illustrated in Fig. 6. We set a Shapley value threshold of 0.2 and selected the top 10 critical bonds along with their corresponding neighbor bonds based on their occurrence frequency. The substructure bonds were highlighted based on the Shapley value. In Fig. 6a, the top 10 functional groups, which were influenced stability, represented by phenyl ring, ring, trifluoromethylbenzene, monofluorobenzene, chlorobenzene, ether, trifluoromethyl, primary amine, alkylbenzene, and carbonyl group have been screened out. They respectively account for approximately 17.9%, 8.2%, 4.5%, 4.4%, 4.2%, 1.2%, 1.1%, 0.9%, 0.9%, and 0.8% of instances where the main bond Shapley value exceeds 0.2. The functional groups that contribute strongly to stability are primarily the top five, accounting for 39.2% of instances, and trifluoromethylbenzene generally has a high Shapley value. Our results are essentially consistent with the literature (Wojtuch et al. 2021) and sequence-based interpretability.\\n\\nInterpretability case study of crucial functional groups for stable and unstable molecules based on graph structures. (a) The statistical results of the top 10 functional groups with significant impact on stability are arranged in descending order according to their frequency of occurrence. In the substructures, red indicates a positive influence, while blue indicates a negative influence. (b) The statistical results of the top 10 functional groups with significant impact on instability are arranged in descending order of occurrence. In these substructures, red signifies a positive influence, while blue signifies a negative influence.\\n\\nIn Fig. 6b, the top 10 functional groups most prone to metabolic instability are sulfonate, amide, secondary amine, diazaheterocycle, primary amine, diazabenzene, amide or carboxylic acid, trifluorobenzene, and sulfonate, with respective proportions of 6.6%, 5.8%, 5.6%, 5.2%, 3.4%, 3.0%, 2.7%, 2.7%, 2.5%, and 2.4% of instances where the main bond Shapley value exceeds 0.2. Due to the extensive variety of metabolic pathway types (Du et al. 2022), there are many functional groups influencing metabolic instability. Consequently, the distribution of these functional groups tends to be quite dispersed. Our results are essentially consistent with the literature (Wojtuch et al. 2021) and sequence-based interpretability, with the exception of trifluorobenzene. The occurrence probability of trifluorobenzene in metabolic instability is significantly lower than its occurrence in the cases of stability. This discrepancy may potentially be attributed to statistical errors.\\n\\nIn summary, the alignment of our findings from the model and existing literature demonstrates the interpretability of CMMS-GCL, highlighting functional groups significantly associated with metabolic stability. This understanding enables the elucidation of the reasons behind a molecule’s specific stability or instability properties. Meanwhile, the insights derived from our analysis can contribute to the guidance of molecular design and the optimization of lead compounds.\\n\\n4 Discussion and conclusion\\n\\nAssessing metabolic stability is essential for drug discovery and development. However, the high cost and risk associated with wet-lab experiments make it challenging to conduct this assessment. Therefore, computational methods can be a fast and effective complementary approach for predicting metabolic stability. However, it is noted that few computational methods have been developed for such tasks.\\n\\nIn this article, we present CMMS-GCL, a novel cross-modality graph contrastive learning model named CMMS-GCL for predicting the metabolic stability of compound candidates. In the model, we design a dual-channel strategy to learn representations for molecules from two different modalities, i.e. SMILES sequence and graph structure. Especially, with the SMILES sequence data as inputs, a multihead BiGRU-based encoder is designed to learn the sequence representations of molecules by fully preserving the local chemical context of atoms. The introduction of multihead BiGRU enables the identification of crucial functional groups specific to metabolic stability. Meanwhile, we propose a GIN-based encoder to learn structure representations from the molecule structure, where graph contrastive learning is further introduced to enhance representation learning by capturing dependencies between local and global structures. Comprehensive experimental results on two datasets demonstrate that our proposed CMMS-GCL outperforms seven state-of-the-art methods in predicting metabolic stability. Additionally, we performed case studies on various stable and unstable molecules, demonstrating the interpretability of key functional groups recognized by CMMS-GCL in sequence analysis. Moreover, we conducted a statistical interpretability analysis utilizing the graph structure, further emphasizing the advantages and potential utility of our method. These statistical analyses not only corroborate our model but also provide valuable guidance for future candidate drug screening and lead compound optimization.\\n\\nAlthough the CMMS-GCL method shows promising results in predicting metabolic stability, it still faces several limitations, including the challenges in effectively incorporating unlabeled data with experimental data and the discrepancies between in vitro data and in vivo predictions due to factors such as complex in vivo metabolism, physiological factors, and interindividual variability. To address these challenges, semisupervised and self-supervised learning techniques can be designed to integrate unlabeled data with the existing labeled data, thereby enhancing learning accuracy and overall prediction performance. Moreover, the incorporation of diverse data sources, validation of model performance using in vivo data, and exploration of methodologies to account for physiological factors and interindividual variability can also be investigated in future research.\\n\\nAcknowledgements\\n\\nThe authors would like to thank anonymous reviewers for suggestions that improved the article.\\n\\nContributor Information\\n\\nBing-Xue Du, School of Life Sciences, Northwestern Polytechnical University, Xi’an 710072, China; Institute for Infocomm Research (I2R), Agency for Science, Technology and Research (A*STAR), Singapore 138632, Singapore.\\n\\nYahui Long, Singapore Immunology Network (SIgN), Agency for Science, Technology and Research (A*STAR), Singapore 138648, Singapore.\\n\\nXiaoli Li, Institute for Infocomm Research (I2R), Agency for Science, Technology and Research (A*STAR), Singapore 138632, Singapore.\\n\\nMin Wu, Institute for Infocomm Research (I2R), Agency for Science, Technology and Research (A*STAR), Singapore 138632, Singapore.\\n\\nJian-Yu Shi, School of Life Sciences, Northwestern Polytechnical University, Xi’an 710072, China.\\n\\nConflict of interest\\n\\nNone declared.\\n\\nFunding\\n\\nThis work was supported in part by the National Nature Science Foundation of China [61872297], Shaanxi Province Key R&D Program [2023-YBSF-114], and CAAI-Huawei MindSpore Open Fund [CAAIXSJLJJ-2022-035A].\\n\\nReferences\\n"}], "questions": [{"id": "openbabel-format-coverage", "question": "According to the paper, what did Open Babel version 2.3 interconvert?", "answerable": true, "expected_document_ids": ["openbabel-2011"], "reference_answer": "It interconverted over 110 chemical file formats.", "source_anchor": "Abstract > Results", "source_quote": "Open Babel version 2.3 interconverts over 110 formats."}, {"id": "openbabel-utilities", "question": "Name two utilities, other than format conversion, that Open Babel provides according to the paper.", "answerable": true, "expected_document_ids": ["openbabel-2011"], "reference_answer": "Conformer searching and 2D depiction. Other supported examples include filtering and substructure or similarity searching.", "source_anchor": "Abstract > Conclusions", "source_quote": "a variety of useful utilities from conformer searching and 2D depiction"}, {"id": "cmms-modalities", "question": "Which two molecular modalities does CMMS-GCL use to extract molecular features?", "answerable": true, "expected_document_ids": ["cmms-gcl-2023"], "reference_answer": "SMILES sequences and molecular graphs.", "source_anchor": "Abstract > Results", "source_quote": "two modality data, i.e. SMILES sequence and molecule graph"}, {"id": "cmms-task", "question": "What prediction task was CMMS-GCL developed to address?", "answerable": true, "expected_document_ids": ["cmms-gcl-2023"], "reference_answer": "Prediction of the metabolic stability of drug candidates.", "source_anchor": "Abstract > Results", "source_quote": "CMMS-GCL for predicting the metabolic stability of drug candidates"}, {"id": "joint-protocol-insufficient", "question": "What shared train, validation, and test split should be used for a joint benchmark of Open Babel and CMMS-GCL?", "answerable": false, "expected_document_ids": ["openbabel-2011", "cmms-gcl-2023"], "reference_answer": "Insufficient information: neither paper defines a shared benchmark or joint train/validation/test split across these two systems.", "source_anchor": "Corpus-level negative control; not specified in either selected article.", "source_quote": null}], "answers": [{"id": "openbabel-format-coverage", "question": "According to the paper, what did Open Babel version 2.3 interconvert?", "answer": "Open Babel version 2.3 interconverted over 110 chemical and molecular data formats (openbabel-2011 chunk 1). This capability is supported by a library implementing a wide range of cheminformatics algorithms, including partial charge assignment, aromaticity detection, bond order perception, and canonicalization (openbabel-2011 chunk 1). The system\'s modular architecture allows for the addition of new formats and plugins without modifying the core code (openbabel-2011 chunk 5). While the exact list of formats is not specified in the excerpt, the emphasis is on the ability to handle a broad spectrum of chemical data types, facilitating tasks such as conformer searching, 2D depiction, filtering, batch conversion, and substructure and similarity searching (openbabel-2011 chunk 1). The modular design ensures flexibility and extensibility, enabling the integration of various functionalities like fingerprints and forcefields through plugins (openbabel-2011 chunk 5). The testing of version 2.3.1 highlighted improvements in handling specific formats, particularly SMILES and MDL Molfile, with significant reductions in error rates for stereochemistry perception, kekulization, and canonicalization (openbabel-2011 chunk 6). These enhancements underscore the robustness and reliability of Open Babel in managing diverse chemical data formats. The system\'s ability to interconvert between formats is a critical feature, addressing the challenges posed by the proliferation of chemical file formats (openbabel-2011 chunk 1).", "references": "1. (openbabel-2011 chunk 1): O\'Boyle NM, Banck M, James CA, Morley C, Vandermeersch T, Hutchison GR. Open Babel: An open chemical toolbox. J Cheminform. 2011;3:33. doi:10.1186/1758-2946-3-33.\\n\\n2. (openbabel-2011 chunk 5): O\'Boyle NM, Banck M, James CA, Morley C, Vandermeersch T, Hutchison GR. Open Babel: An open chemical toolbox. J Cheminform. 2011;3:33. doi:10.1186/1758-2946-3-33.\\n\\n3. (openbabel-2011 chunk 6): O\'Boyle NM, Banck M, James CA, Morley C, Vandermeersch T, Hutchison GR. Open Babel: An open chemical toolbox. J Cheminform. 2011;3:33. doi:10.1186/1758-2946-3-33.", "elapsed_seconds": 27.723, "contexts": [{"document_id": "openbabel-2011", "chunk_name": "openbabel-2011 chunk 5", "source_text": " using the CMake build system [37,38]. This is an open-source cross-platform build system with advanced features for dependency analysis. The build system has an associated unit test framework CTest, which allows nightly builds to be compiled and tested automatically with the results collated and displayed on a centralized dashboard [39].\\n\\nTo simplify installation Open Babel has as few external dependencies as possible. Where such dependencies exist, they are optional. For example, if the XML development libraries are not available, Open Babel will still compile successfully but none of the XML formats (such as Chemical Markup Language, CML) will be available. Similarly, if the Eigen matrix and linear algebra library is not found, any classes that require fast matrix manipulation (such as OBAlign, which performs least squares alignment) will not be compiled.\\n\\nWhile the majority of the Open Babel library is written in C++, bindings have been developed for a range of other programming languages, including Java and the .NET platform, as well as the so-called \\"dynamic\\" scripting languages Perl, Python, and Ruby. These are automatically generated from the C++ header files using the SWIG tool. As described previously [40], in the case of Python an additional module is provided named Pybel that simplifies access to the C++ bindings. These interfaces facilitate development of web-enabled chemistry applications, as well as rapid development and prototyping.\\n\\nCode Architecture\\n\\nThe Open Babel codebase has a modular design as shown in Figure 2. The goal of this design is threefold:\\n\\nArchitecture of the Open Babel codebase.\\n\\n1. To separate the chemistry, the conversion process and the user interfaces reducing, as far as possible, the dependency of one upon another.\\n\\n2. To put all of the code for each chemical format in one place (usually a single file) and make the addition of new formats simple.\\n\\n3. To allow the format conversion of not just molecules, but also any other chemical objects, such as reactions.\\n\\nThe code base can be considered as consisting of the following modules (Figure 2):\\n\\n• The Chemical Core, which contains OBMol etc. and has all of the chemical structure description and manipulation. This is the heart of the application and its API can be used as a chemical toolbox. It has no input/output capabilities.\\n\\n• The Formats, which read and write to files of different types. These classes are derived from a common base class, OBFormat, which is in the Conversion Control module. They also make use of the chemical routines in the Chemical Core module. Each format file contains a global object of the format class. When the format is loaded the class constructor registers the presence of the class with OBConversion. This means that the formats are plugins - new formats can be added without changing any framework code.\\n\\n• Common Formats include OBMoleculeFormat and XMLBaseFormat from which most other formats (like Format A and Format B in the diagram) are derived. Independent formats like Format C are also possible.\\n\\n• The Conversion Control, which also keeps track of the available formats, the conversion options and the input and output streams. It can be compiled without reference to any other parts of the program. In particular, it knows nothing of the Chemical Core: mol.h is not included.\\n\\n• The User Interface, which may be a command line application, a Graphical User Interface (GUI), or may be part of another program that uses Open Babel\'s input and output facilities. This depends only on the Conversion Control module (obconversion.h is included), but not on the Chemical Core or on any of the Formats.\\n\\n• The Fingerprint API, as well as being usable in external programs, is employed by the fastsearch and fingerprint formats.\\n\\n• The Fingerprints, which are bit arrays that describe an object and which facilitate fast searching. They are also built as plugins, registering themselves with their base class OBFingerprint which is in the Fingerprint API.\\n\\n• Other features such as Forcefields, Partial Charge Models and Chemical Descriptors, although not shown in the diagram, are handled similarly to Fingerprints.\\n\\n• The Error Handling can be used throughout the program to log and display errors and warnings.\\n\\nExtensible Interface\\n\\nThe utility of software libraries such as Open Babel depends on the ability of the design to be extended over time to support new functionality. To facilitate this, Open Babel implements a plugin interface for file formats, fingerprints, charge models, descriptors, \\"operators\\" and molecular mechanics force fields. This ensures a clean separation of the implementation of a particular plugin from the core Open Babel library code, and makes it easy for a new plugin (e.g. a new file format) to be contributed; all that is needed is a single C++ file and a trivial change to one of the build files. The operator plugins provide a very general mechanism for operating on a molecule (e.g. energy minimization or 3D coordinate generation) or on a list of molecules (e.g. filtering or sorting) after reading but before writing.\\n\\nPlugins are dynamically loaded at runtime. This decreases the overall disk and memory footprint of Open Babel, allowing external developers to choose particular functionality needed for their application and ignore other, less relevant features. It also allows the possibility of a third-party distributing plugins separately to the Open Babel distribution to provide additional functionality.\\n\\nOpen-Source License and Open Development\\n\\nOpen Babel is open-source software, which offers end users and third-party developers a range of additional rights not granted by proprietary chemistry software. Open-source software, at its most basic level, grants users the rights to study how their", "summary": "The excerpt discusses the features and architecture of Open Babel, an open-source chemical toolbox. It mentions that Open Babel supports a wide range of chemical formats and can interconvert between them. The codebase is modular, with separate modules for chemical core, formats, conversion control, user interfaces, and plugins for various functionalities like fingerprints and forcefields. The system allows for the addition of new formats and plugins without modifying the core code. However, the excerpt does not specify the exact formats interconverted by Open Babel version 2.3.", "relevance": 3, "cited": true}, {"document_id": "openbabel-2011", "chunk_name": "openbabel-2011 chunk 6", "source_text": " footprint of Open Babel, allowing external developers to choose particular functionality needed for their application and ignore other, less relevant features. It also allows the possibility of a third-party distributing plugins separately to the Open Babel distribution to provide additional functionality.\\n\\nOpen-Source License and Open Development\\n\\nOpen Babel is open-source software, which offers end users and third-party developers a range of additional rights not granted by proprietary chemistry software. Open-source software, at its most basic level, grants users the rights to study how their software works, to adapt it for any purpose or otherwise modify it, and to share the software and their modifications with others. In this sense, Open Source functions in similar ways to the processes of open peer review, publication, and citation in science. The rights granted by open source licenses largely coincide with the norms of scientific ethics to enable verifiability, repeatability, and building on previous results and theories.\\n\\nBeyond these rights, Open Babel (like most other open-source projects) offers open development -- that is, all development occurs in public forums and with public code repositories. This results in greater input from the community as any user can easily submit bug reports or feature suggestions, get involved in discussions on the future direction of Open Babel or even become a developer him/herself. In practice, the number of active contributors has increased over time through this level of open, public development (Figure 3). Moreover, it means that the development of the code is completely transparent and the quality of the software is available for public scrutiny. Indeed, since its inception, over 658 bugs have been submitted to the public tracker and fixed [41].\\n\\nNumber of contributors over time. Note that this graph only includes developers who directly commited code to the Open Babel source code repository, and does not include patches provided by users.\\n\\nValidation and Testing\\n\\nOpen Babel includes an extensive test suite comprising 60 different test programs each with tens to hundreds of tests. In early 2010, a nightly build infrastructure and dashboard was put in place with support from Kitware, Inc. This has greatly improved code quality by catching regressions, and also ensures that the code compiles cleanly on all platforms and compilers supported by Open Babel. Some examples of tests that are run each night are:\\n\\n(1) The MMFF94 forcefield code is tested against the MMFF94 validation suite.\\n\\n(2) The OBAlign class, which was developed using Test-Driven Development (TDD) methodology, is run against its test suite.\\n\\n(3) Handling of symmetry is validated by converting several test cases between SMILES, 2D and 3D SDF, and InChI (there are also several test programs with unit tests for the individual stereo classes in the API).\\n\\n(4) The SMARTS parser is tested using over 250 valid and invalid SMARTS patterns, and the SMARTS matcher is tested using 125 basic SMARTS patterns.\\n\\n(5) The LSSR (Least Set of Smallest Rings) code is tested for invariance against changing the atom order for a series of polycyclic molecules.\\n\\nRecently the development team has placed a major focus on increasing the robustness of file format translation particularly in relation to the commonly used SMILES and MDL Molfile formats. Translating between these formats requires accurate stereochemistry perception, inference of implicit hydrogens, and kekulization of delocalized systems. While it is difficult to ensure that any complex piece of code is free of bugs, and Open Babel is no exception, validation procedures can be carried out to assess the current level of performance and to find additional test cases that expose bugs. The following procedure was used to guide the rewriting of stereochemistry code in Open Babel, a project that began in early 2009. Starting with a dataset of 18,084 3D structures from PubChem3D as an SDF file, we compared the result of (a) conversion to SMILES, followed by conversion of that to Canonical SMILES to (b) conversion directly to Canonical SMILES. This procedure can be used to flush out errors in reading the original SDF file, reading/writing SMILES (either due to stereochemistry errors or kekulization problems), and is also a test (to some extent) of the canonicalization code. At the time of starting this work (March 2009), the error rate found was 1424 (8%); by Oct 2009, combined work on stereochemistry, kekulization and canonicalization had reduced this to 190 (~1%), and continued improvements have reduced the number of errors down to two (shown in Figure 4) for Open Babel 2.3.1 (~0.01%). The first failure is due to a kekulization error in a polycyclic aromatic molecule incorporating heteroatoms: (a) gave c1ccc2c(c1)c1[nH][nH]c3c4c1c(c2)ccc4cc1c3cccc1 while (b) gave c1ccc2c(c1)c1nnc3c4c1c(c2)ccc4cc1c3cccc1. This error led to confusion over whether or not the aromatic nitrogens have hydrogens attached (they do not). The second failure involves confusion over the canonical stereochemistry at a bridgehead carbon: (a) gave C1CN2[C@@H](C1)CCC2 while (b) gave C1CN2[C@H](", "summary": "According to the paper, Open Babel version 2.3.1 was tested for its ability to interconvert between different chemical formats, particularly focusing on SMILES and MDL Molfile formats. The testing involved comparing the results of converting 3D structures from PubChem3D to SMILES and then to Canonical SMILES, versus converting directly to Canonical SMILES. This process helped identify and reduce errors in stereochemistry perception, kekulization, and canonicalization. The error rate was significantly reduced from 8% to ~0.01% through continuous improvements.", "relevance": 10, "cited": true}, {"document_id": "openbabel-2011", "chunk_name": "openbabel-2011 chunk 1", "source_text": "Open Babel: An open chemical toolbox\\n\\nO\'Boyle NM, Banck M, James CA, Morley C, Vandermeersch T, Hutchison GR. Open Babel: An open chemical toolbox. J Cheminform. 2011;3:33. doi:10.1186/1758-2946-3-33.\\n\\nhttps://creativecommons.org/licenses/by/2.0/\\n\\nAbstract\\n\\nBackground\\n\\nA frequent problem in computational modeling is the interconversion of chemical structures between different formats. While standard interchange formats exist (for example, Chemical Markup Language) and de facto standards have arisen (for example, SMILES format), the need to interconvert formats is a continuing problem due to the multitude of different application areas for chemistry data, differences in the data stored by different formats (0D versus 3D, for example), and competition between software along with a lack of vendor-neutral formats.\\n\\nResults\\n\\nWe discuss, for the first time, Open Babel, an open-source chemical toolbox that speaks the many languages of chemical data. Open Babel version 2.3 interconverts over 110 formats. The need to represent such a wide variety of chemical and molecular data requires a library that implements a wide range of cheminformatics algorithms, from partial charge assignment and aromaticity detection, to bond order perception and canonicalization. We detail the implementation of Open Babel, describe key advances in the 2.3 release, and outline a variety of uses both in terms of software products and scientific research, including applications far beyond simple format interconversion.\\n\\nConclusions\\n\\nOpen Babel presents a solution to the proliferation of multiple chemical file formats. In addition, it provides a variety of useful utilities from conformer searching and 2D depiction, to filtering, batch conversion, and substructure and similarity searching. For developers, it can be used as a programming library to handle chemical data in areas such as organic chemistry, drug design, materials science, and computational chemistry. It is freely available under an open-source license from http://openbabel.org.\\n\\nIntroduction\\n\\nThe history of chemical informatics has included a huge variety of textual and computer representations of molecular data. Such representations focus on specific atomic or molecular information and may not attempt to store all possible chemical data. For example, line notations like Daylight SMILES [1] do not offer coordinate information, while crystallographic or quantum mechanical formats frequently do not store chemical bonding data. Hydrogen atoms are frequently omitted from x-ray crystallography due to the difficulty in establishing coordinates, and are often ignored by some file formats as the \\"implicit valence\\" of heavy atoms that indicates their presence. Other types of representations require specification of atom types on the basis of a specific valence bond model, inclusion of computed partial charges, indication of biomolecular residues, or multiple conformations.\\n\\nWhile attempts have been made to provide a standard format for storing chemical data, including most notably the development of Chemical Markup Language (CML) [2-6], an XML dialect, such formats have not yet achieved widespread use. Consequently, a frequent problem in computational modeling is the interconversion of molecular structures between different formats, a process that involves extraction and interpretation of their chemical data and semantics.\\n\\nWe outline for the first time, the development and use of the Open Babel project, a full-featured open chemical toolbox, designed to \\"speak\\" the many different representations of chemical data. It allows anyone to search, convert, analyze, or store data from molecular modeling, chemistry, solid-state materials, biochemistry, or related areas. It provides both ready-to-use programs as well as a complete, extensible programmer\'s toolkit for developing cheminformatics software. It can handle reading, writing, and interconverting over 110 chemical file formats, supports filtering and searching molecule files using Daylight SMARTS pattern matching [7] and other methods, and provides extensible fingerprinting and molecular mechanics frameworks. We will discuss the frameworks for file format interconversion, fingerprinting, fast molecular searching, bond perception and atom typing, canonical numbering of molecular structures and fragments, molecular mechanics force fields, and the extensible interfaces provided by the software library to enable further chemistry software development.\\n\\nOpen Babel has its origin in a version of OELib released as open-source software by OpenEye Scientific under the GPL (GNU Public License). In 2001, OpenEye decided to rewrite OELib in-house as the proprietary OEChem library, so the existing code from OELib was spun out into the new Open Babel project. Since 2001, Open Babel has been developed and substantially extended as an international collaborative project using an open-source development model [8]. It has over 160,000 downloads, over 400 citations [9], is used by over 40 software projects [10], and is freely available from the Open Babel website [11].\\n\\nFeatures\\n\\nFile Format Support\\n\\nWith the release of Open Babel 2.3, Open Babel supports 111 chemical file formats in total. It can read 82 formats and write 85 formats. These encompass common formats used in cheminformatics (SMILES, InChI, MOL, MOL2), input and output files from a variety of computational chemistry packages (GAMESS, Gaussian, M", "summary": "According to the paper, Open Babel version 2.3 interconverts over 110 formats. The need to represent such a wide variety of chemical and molecular data requires a library that implements a wide range of cheminformatics algorithms, from partial charge assignment and aromaticity detection, to bond order perception and canonicalization. Open Babel provides a solution to the proliferation of multiple chemical file formats and offers a variety of useful utilities from conformer searching and 2D depiction, to filtering, batch conversion, and substructure and similarity searching.", "relevance": 10, "cited": true}, {"document_id": "openbabel-2011", "chunk_name": "openbabel-2011 chunk 8", "source_text": " Babel to the user for use in programs. Documentation on the complete API (generated using Doxygen [42]) is available from the Open Babel website [43], or can be generated from the source code.\\n\\nThe functionality provided by the Open Babel library is relied upon by many users and by several other software projects, with the result that introducing changes to the API would cause existing software to break. For this reason, Open Babel strives to maintain API stability over long periods of time, so that existing software will continue to work despite the release of new Open Babel versions with additional features, file formats and bug fixes. Open Babel uses a version numbering system that indicates how the API has changed with every release:\\n\\n• Bug fix releases (e.g. 2.0.0 versus 2.0.1) do not change API at all\\n\\n• Minor version releases (e.g. 2.0 versus 2.1) will add to the API, but will otherwise be backwards-compatible\\n\\n• Major version releases (e.g. 2 versus 3) are not backwards-compatible, and have changes to the API (including removal of deprecated classes and functions)\\n\\nFigure 7 shows an example C++ program that uses the two main classes OBConversion and OBMol to print out the molecular weight of all of the molecules in an SDF file. This could be used, for example, to investigate differences in the molecular weight distribution between two databases. The same program is shown in Figure 8 but implemented using the Python bindings.\\n\\nExample C++ program that uses the Open Babel library. The program prints out the molecular weight of each molecule in the SDF file \\"dataset.sdf\\".\\n\\nExample Python program that uses the Open Babel library. The program prints out the molecular weight of each molecule in the SDF file \\"dataset.sdf\\".\\n\\nExamples of Use\\n\\nOpen Babel has already been referenced over 400 times for various uses. The most common use of Open Babel is through the obabel command line application (or the corresponding graphical user interface) for the interconversion of chemical file formats. Such conversions may also involve the calculation or inference of additional molecular information or application of a filter. Some published examples of these include the following:\\n\\n• interconversion of chemical file formats or representations [44-47]\\n\\n• addition of hydrogens [48-50]\\n\\n• generation of 3D molecular structures [51-53]\\n\\n• calculation of partial charges [54,55]\\n\\n• generation of molecular fingerprints [56-59]\\n\\n• removal of duplicate molecules from a dataset [60]\\n\\n• calculation of MOL2 atom types [61]\\n\\nAn interesting example that shows how a particular chemical representation may be used to facilitate a scientific study is the crystallographic study of Fábián and Brock who used Open Babel to generate InChI strings for molecules in the Cambridge Structural Database [62]. Exploiting the fact that InChIs of enantiomers are identical expect at the enantiomer sublayer (\\"/m0\\" or \\"/m1\\"), they used the InChIs as part of a workflow to identify kryptoracemates (a class of racemic crystals where the enantiomers are not related by space-group symmetry) in the database.\\n\\nTo implement new methods, or access additional molecular information, it is necessary to use the Open Babel library directly either from C++ or using one of the supported language bindings. Some examples of published studies that have done this include the following:\\n\\n• Dehmer et al. implemented molecular complexity measures based on information theory [63].\\n\\n• Langham and Jain developed a model for chemical mutagenicity based on atom pair features [64].\\n\\n• Fontaine et al. implemented a method, anchor-GRIND, that uses an anchor point of a molecular scaffold to compare molecular interaction fields when different substituents are present [65].\\n\\n• Konyk et al. have developed a plugin for Open Babel that adds support for the Web Ontology Language (OWL) to allow automated reasoning about chemical structures [66].\\n\\n• Kogej et al. (AstraZeneca) implemented a 3-point pharmacophore fingerprint called TRUST [67].\\n\\n• Many other examples exist [68-71].\\n\\nThe vital role that a cheminformatics toolkit plays in the development of scientific resources is shown by Tables 1 and 2. Table 1 lists examples of stand-alone applications or programming libraries that rely on Open Babel, either calling the library directly or via one of the command-line executables. Table 2 contains examples of web applications and databases that either use Open Babel on the server or where Open Babel was used in the preparation of the data.\\n\\nSoftware applications and libraries that use Open Babel\\n\\nWeb applications and databases that use Open Babel\\n\\nConclusions\\n\\nIn November 2011, Open Babel will mark 10 years of existence as an independent project, and for the first time, we have discussed its development and features. As shown by more than 400 citations, it has become an essential tool for handling the myriad of molecular file formats encountered in diverse branches of chemistry. While more work remains to be done, through validation processes such as those described above and the recent introduction of a nightly build and testing framework, we aim to improve the quality and robustness of the toolkit with each new release.\\n\\nLooking forward to the future, one of the goals of the project is to extend support to molecules that currently are not handled very well by existing cheminformatics toolkits. Typically toolkits focus on the", "summary": "The provided excerpt discusses the functionalities and applications of Open Babel, a cheminformatics toolkit. It mentions that Open Babel is used for interconversion of chemical file formats, addition of hydrogens, generation of 3D molecular structures, calculation of partial charges, generation of molecular fingerprints, removal of duplicate molecules, and calculation of MOL2 atom types. The text also references various studies and applications that use Open Babel, including the conversion of chemical file formats and the generation of InChI strings for molecular analysis. However, it does not specifically mention what Open Babel version 2.3 interconverted.", "relevance": 2, "cited": false}]}, {"id": "openbabel-utilities", "question": "Name two utilities, other than format conversion, that Open Babel provides according to the paper.", "answer": "Open Babel provides several utilities beyond format conversion. Two such utilities are the fingerprint format, which is used for similarity and substructure searching, and the fastsearch format, which is also used for similarity and substructure searching (openbabel-2011 chunk 2). These formats leverage the same interface as regular file formats, enabling functionalities such as generating molecular summaries and performing efficient searches in chemical databases. Additionally, the Fingerprint API is another utility that generates bit arrays to describe chemical objects, facilitating fast searching (openbabel-2011 chunk 5). This API enhances the functionality of Open Babel by enabling efficient data processing. Another utility is the MMFF94 forcefield code, which is tested against the MMFF94 validation suite, demonstrating the software\'s capabilities in molecular modeling (openbabel-2011 chunk 6). The OBAlign class, developed using Test-Driven Development (TDD) methodology, is another utility that is run against its test suite, further showcasing the software\'s robustness in molecular alignment. These utilities collectively illustrate the versatility and advanced features of Open Babel beyond basic format conversion, supporting its role as an open chemical toolbox.", "references": "1. (openbabel-2011 chunk 2): O\'Boyle NM, Banck M, James CA, Morley C, Vandermeersch T, Hutchison GR. Open Babel: An open chemical toolbox. J Cheminform. 2011;3:33. doi:10.1186/1758-2946-3-33.\\n\\n2. (openbabel-2011 chunk 5): O\'Boyle NM, Banck M, James CA, Morley C, Vandermeersch T, Hutchison GR. Open Babel: An open chemical toolbox. J Cheminform. 2011;3:33. doi:10.1186/1758-2946-3-33.\\n\\n3. (openbabel-2011 chunk 6): O\'Boyle NM, Banck M, James CA, Morley C, Vandermeersch T, Hutchison GR. Open Babel: An open chemical toolbox. J Cheminform. 2011;3:33. doi:10.1186/1758-2946-3-33.", "elapsed_seconds": 15.221, "contexts": [{"document_id": "openbabel-2011", "chunk_name": "openbabel-2011 chunk 5", "source_text": " using the CMake build system [37,38]. This is an open-source cross-platform build system with advanced features for dependency analysis. The build system has an associated unit test framework CTest, which allows nightly builds to be compiled and tested automatically with the results collated and displayed on a centralized dashboard [39].\\n\\nTo simplify installation Open Babel has as few external dependencies as possible. Where such dependencies exist, they are optional. For example, if the XML development libraries are not available, Open Babel will still compile successfully but none of the XML formats (such as Chemical Markup Language, CML) will be available. Similarly, if the Eigen matrix and linear algebra library is not found, any classes that require fast matrix manipulation (such as OBAlign, which performs least squares alignment) will not be compiled.\\n\\nWhile the majority of the Open Babel library is written in C++, bindings have been developed for a range of other programming languages, including Java and the .NET platform, as well as the so-called \\"dynamic\\" scripting languages Perl, Python, and Ruby. These are automatically generated from the C++ header files using the SWIG tool. As described previously [40], in the case of Python an additional module is provided named Pybel that simplifies access to the C++ bindings. These interfaces facilitate development of web-enabled chemistry applications, as well as rapid development and prototyping.\\n\\nCode Architecture\\n\\nThe Open Babel codebase has a modular design as shown in Figure 2. The goal of this design is threefold:\\n\\nArchitecture of the Open Babel codebase.\\n\\n1. To separate the chemistry, the conversion process and the user interfaces reducing, as far as possible, the dependency of one upon another.\\n\\n2. To put all of the code for each chemical format in one place (usually a single file) and make the addition of new formats simple.\\n\\n3. To allow the format conversion of not just molecules, but also any other chemical objects, such as reactions.\\n\\nThe code base can be considered as consisting of the following modules (Figure 2):\\n\\n• The Chemical Core, which contains OBMol etc. and has all of the chemical structure description and manipulation. This is the heart of the application and its API can be used as a chemical toolbox. It has no input/output capabilities.\\n\\n• The Formats, which read and write to files of different types. These classes are derived from a common base class, OBFormat, which is in the Conversion Control module. They also make use of the chemical routines in the Chemical Core module. Each format file contains a global object of the format class. When the format is loaded the class constructor registers the presence of the class with OBConversion. This means that the formats are plugins - new formats can be added without changing any framework code.\\n\\n• Common Formats include OBMoleculeFormat and XMLBaseFormat from which most other formats (like Format A and Format B in the diagram) are derived. Independent formats like Format C are also possible.\\n\\n• The Conversion Control, which also keeps track of the available formats, the conversion options and the input and output streams. It can be compiled without reference to any other parts of the program. In particular, it knows nothing of the Chemical Core: mol.h is not included.\\n\\n• The User Interface, which may be a command line application, a Graphical User Interface (GUI), or may be part of another program that uses Open Babel\'s input and output facilities. This depends only on the Conversion Control module (obconversion.h is included), but not on the Chemical Core or on any of the Formats.\\n\\n• The Fingerprint API, as well as being usable in external programs, is employed by the fastsearch and fingerprint formats.\\n\\n• The Fingerprints, which are bit arrays that describe an object and which facilitate fast searching. They are also built as plugins, registering themselves with their base class OBFingerprint which is in the Fingerprint API.\\n\\n• Other features such as Forcefields, Partial Charge Models and Chemical Descriptors, although not shown in the diagram, are handled similarly to Fingerprints.\\n\\n• The Error Handling can be used throughout the program to log and display errors and warnings.\\n\\nExtensible Interface\\n\\nThe utility of software libraries such as Open Babel depends on the ability of the design to be extended over time to support new functionality. To facilitate this, Open Babel implements a plugin interface for file formats, fingerprints, charge models, descriptors, \\"operators\\" and molecular mechanics force fields. This ensures a clean separation of the implementation of a particular plugin from the core Open Babel library code, and makes it easy for a new plugin (e.g. a new file format) to be contributed; all that is needed is a single C++ file and a trivial change to one of the build files. The operator plugins provide a very general mechanism for operating on a molecule (e.g. energy minimization or 3D coordinate generation) or on a list of molecules (e.g. filtering or sorting) after reading but before writing.\\n\\nPlugins are dynamically loaded at runtime. This decreases the overall disk and memory footprint of Open Babel, allowing external developers to choose particular functionality needed for their application and ignore other, less relevant features. It also allows the possibility of a third-party distributing plugins separately to the Open Babel distribution to provide additional functionality.\\n\\nOpen-Source License and Open Development\\n\\nOpen Babel is open-source software, which offers end users and third-party developers a range of additional rights not granted by proprietary chemistry software. Open-source software, at its most basic level, grants users the rights to study how their", "summary": "Open Babel provides several utilities beyond format conversion. Two of these utilities are the Fingerprint API and the Error Handling system. The Fingerprint API is used for generating bit arrays that describe chemical objects, facilitating fast searching. The Error Handling system is used throughout the program to log and display errors and warnings. These utilities enhance the functionality of Open Babel by enabling efficient data processing and robust error management.", "relevance": 10, "cited": true}, {"document_id": "openbabel-2011", "chunk_name": "openbabel-2011 chunk 8", "source_text": " Babel to the user for use in programs. Documentation on the complete API (generated using Doxygen [42]) is available from the Open Babel website [43], or can be generated from the source code.\\n\\nThe functionality provided by the Open Babel library is relied upon by many users and by several other software projects, with the result that introducing changes to the API would cause existing software to break. For this reason, Open Babel strives to maintain API stability over long periods of time, so that existing software will continue to work despite the release of new Open Babel versions with additional features, file formats and bug fixes. Open Babel uses a version numbering system that indicates how the API has changed with every release:\\n\\n• Bug fix releases (e.g. 2.0.0 versus 2.0.1) do not change API at all\\n\\n• Minor version releases (e.g. 2.0 versus 2.1) will add to the API, but will otherwise be backwards-compatible\\n\\n• Major version releases (e.g. 2 versus 3) are not backwards-compatible, and have changes to the API (including removal of deprecated classes and functions)\\n\\nFigure 7 shows an example C++ program that uses the two main classes OBConversion and OBMol to print out the molecular weight of all of the molecules in an SDF file. This could be used, for example, to investigate differences in the molecular weight distribution between two databases. The same program is shown in Figure 8 but implemented using the Python bindings.\\n\\nExample C++ program that uses the Open Babel library. The program prints out the molecular weight of each molecule in the SDF file \\"dataset.sdf\\".\\n\\nExample Python program that uses the Open Babel library. The program prints out the molecular weight of each molecule in the SDF file \\"dataset.sdf\\".\\n\\nExamples of Use\\n\\nOpen Babel has already been referenced over 400 times for various uses. The most common use of Open Babel is through the obabel command line application (or the corresponding graphical user interface) for the interconversion of chemical file formats. Such conversions may also involve the calculation or inference of additional molecular information or application of a filter. Some published examples of these include the following:\\n\\n• interconversion of chemical file formats or representations [44-47]\\n\\n• addition of hydrogens [48-50]\\n\\n• generation of 3D molecular structures [51-53]\\n\\n• calculation of partial charges [54,55]\\n\\n• generation of molecular fingerprints [56-59]\\n\\n• removal of duplicate molecules from a dataset [60]\\n\\n• calculation of MOL2 atom types [61]\\n\\nAn interesting example that shows how a particular chemical representation may be used to facilitate a scientific study is the crystallographic study of Fábián and Brock who used Open Babel to generate InChI strings for molecules in the Cambridge Structural Database [62]. Exploiting the fact that InChIs of enantiomers are identical expect at the enantiomer sublayer (\\"/m0\\" or \\"/m1\\"), they used the InChIs as part of a workflow to identify kryptoracemates (a class of racemic crystals where the enantiomers are not related by space-group symmetry) in the database.\\n\\nTo implement new methods, or access additional molecular information, it is necessary to use the Open Babel library directly either from C++ or using one of the supported language bindings. Some examples of published studies that have done this include the following:\\n\\n• Dehmer et al. implemented molecular complexity measures based on information theory [63].\\n\\n• Langham and Jain developed a model for chemical mutagenicity based on atom pair features [64].\\n\\n• Fontaine et al. implemented a method, anchor-GRIND, that uses an anchor point of a molecular scaffold to compare molecular interaction fields when different substituents are present [65].\\n\\n• Konyk et al. have developed a plugin for Open Babel that adds support for the Web Ontology Language (OWL) to allow automated reasoning about chemical structures [66].\\n\\n• Kogej et al. (AstraZeneca) implemented a 3-point pharmacophore fingerprint called TRUST [67].\\n\\n• Many other examples exist [68-71].\\n\\nThe vital role that a cheminformatics toolkit plays in the development of scientific resources is shown by Tables 1 and 2. Table 1 lists examples of stand-alone applications or programming libraries that rely on Open Babel, either calling the library directly or via one of the command-line executables. Table 2 contains examples of web applications and databases that either use Open Babel on the server or where Open Babel was used in the preparation of the data.\\n\\nSoftware applications and libraries that use Open Babel\\n\\nWeb applications and databases that use Open Babel\\n\\nConclusions\\n\\nIn November 2011, Open Babel will mark 10 years of existence as an independent project, and for the first time, we have discussed its development and features. As shown by more than 400 citations, it has become an essential tool for handling the myriad of molecular file formats encountered in diverse branches of chemistry. While more work remains to be done, through validation processes such as those described above and the recent introduction of a nightly build and testing framework, we aim to improve the quality and robustness of the toolkit with each new release.\\n\\nLooking forward to the future, one of the goals of the project is to extend support to molecules that currently are not handled very well by existing cheminformatics toolkits. Typically toolkits focus on the", "summary": "According to the paper, Open Babel provides utilities beyond format conversion, such as the generation of 3D molecular structures and the calculation of partial charges. Additionally, it supports the generation of molecular fingerprints and the removal of duplicate molecules from a dataset. These functionalities demonstrate the versatility of Open Babel in cheminformatics tasks.", "relevance": 10, "cited": false}, {"document_id": "openbabel-2011", "chunk_name": "openbabel-2011 chunk 2", "source_text": "40 software projects [10], and is freely available from the Open Babel website [11].\\n\\nFeatures\\n\\nFile Format Support\\n\\nWith the release of Open Babel 2.3, Open Babel supports 111 chemical file formats in total. It can read 82 formats and write 85 formats. These encompass common formats used in cheminformatics (SMILES, InChI, MOL, MOL2), input and output files from a variety of computational chemistry packages (GAMESS, Gaussian, MOPAC), crystallographic file formats (CIF, ShelX), reaction formats (MDL RXN), file formats used by molecular dynamics and docking packages (AutoDock, Amber), formats used by 2D drawing packages (ChemDraw), 3D viewers (Chem3D, Molden) and chemical kinetics and thermodynamics (ChemKin, Thermo). Formats are implemented as \\"plugins\\" in Open Babel, which makes it easy for users to contribute new file formats (see Extensible Interface below). Depending on the format, other data is extracted by Open Babel in addition to the molecular structure; for example, vibrational frequencies are extracted from computational chemistry log files, unit cell information is extracted from CIF files, and property fields are read from SDF files.\\n\\nA number of \\"utility\\" file formats are also defined; these are not strictly speaking a way of storing the molecular structure, but rather present certain functionality through the same interface as the regular file formats. For example, the report format is a write-only utility format [12] that presents a summary of the molecular structure of a molecule; the fingerprint format [13] and fastsearch format [14] are used for similarity and substructure searching (see below); the MolPrint2D and Multilevel Neighborhoods of Atoms formats calculate circular fingerprints defined by Bender et al. [15,16] and Filimonov et al. [17,18] respectively.\\n\\nEach format can have multiple options to control either reading or writing a particular format. For example, the InChI format has 12 options including an option \\"K\\" to generate an InChIKey, \\"T <param>\\" to truncate the InChI depending on a supplied parameter and \\"w\\" to ignore certain InChI warnings. The available options are listed in the documentation, are shown in the Graphical User Interface (GUI) as checkboxes or textboxes, and can be listed at the command-line. In fact, all three are generated from the same source; a documentation string in the C++ code.\\n\\nFingerprints and Fast Searching\\n\\nDatabases are widely used to store chemical information especially in the pharmaceutical industry. A key requirement of such a database is the ability to index chemical structures so that they can be quickly retrieved given a query substructure. Open Babel provides this functionality using a path-based fingerprint. This fingerprint, referred to as FP2 in Open Babel, identifies all linear and ring substructures in the molecule of lengths 1 to 7 (excluding the 1-atom substructures C and N) and maps them onto a bit-string of length 1024 using a hash function. If a query molecule is a substructure of a target molecule, then all of the bits set in the query molecule will also be set in the target molecule. The fingerprints for two molecules can also be used to calculate structural similarity using the Tanimoto coefficient, the number of bits in common divided by the union of the bits set.\\n\\nClearly, repeated searching of the same set of molecules will involve repeated use of the same set of fingerprints. To avoid the need to recalculate the fingerprints for a particular multi-molecule file (such as an SDF file), Open Babel provides a fastindex format that solely stores a fingerprint along with an index into the original file. This index leads to a rapid increase in the speed of searching for matches to a query - datasets with several million molecules are easily searched interactively. In this way, a multi-molecule file may be used as a lightweight alternative to a chemical database system.\\n\\nBond Perception and Atom Typing\\n\\nAs mentioned above, many chemical file formats offer representations of molecular data solely as lists of atoms. For example, most quantum chemical software packages and most crystallographic file formats do not offer definitions of bonding. A similar situation occurs in the case of the Protein Data Bank (PDB) format; while standardized [19] files contain connectivity information, non-standard files exist that often do not provide full connectivity information. Consequently, Open Babel features methods to determine bond connectivity, bond order perception, aromaticity determination, and atom typing.\\n\\nBond connectivity is determined by the frequently used algorithm of detecting atoms closer than the sum of their covalent radii, with a slight tolerance (0.45 Å) to allow for longer than typical bonds. To handle disorder in crystallographic data (e.g., PDB or CIF files), atoms closer than 0.63 Å are not bonded. A further filtering pass is made to ensure standard bond valency is maintained; each element has a maximum number of bonds, if this is exceeded then the longest bonds to an atom are successively removed until the valence rule is fulfilled.\\n\\nAfter bond connectivity is determined, if needed or requested by the user, bond order perception is performed on the basis of bond angles and geometries. The method is similar to that proposed by Roger Sayle [20] and uses the average bond angle around an un-typed atom to determine sp and sp2 hybridized centers. 5", "summary": "Open Babel provides several utilities beyond format conversion. Two such utilities are the report format, which is a write-only format that presents a summary of the molecular structure, and the fingerprint format, which is used for similarity and substructure searching. Additionally, the fastsearch format is also used for similarity and substructure searching. These utilities leverage the same interface as regular file formats, enabling functionalities such as generating molecular summaries and performing efficient searches in chemical databases.", "relevance": 10, "cited": true}, {"document_id": "openbabel-2011", "chunk_name": "openbabel-2011 chunk 6", "source_text": " footprint of Open Babel, allowing external developers to choose particular functionality needed for their application and ignore other, less relevant features. It also allows the possibility of a third-party distributing plugins separately to the Open Babel distribution to provide additional functionality.\\n\\nOpen-Source License and Open Development\\n\\nOpen Babel is open-source software, which offers end users and third-party developers a range of additional rights not granted by proprietary chemistry software. Open-source software, at its most basic level, grants users the rights to study how their software works, to adapt it for any purpose or otherwise modify it, and to share the software and their modifications with others. In this sense, Open Source functions in similar ways to the processes of open peer review, publication, and citation in science. The rights granted by open source licenses largely coincide with the norms of scientific ethics to enable verifiability, repeatability, and building on previous results and theories.\\n\\nBeyond these rights, Open Babel (like most other open-source projects) offers open development -- that is, all development occurs in public forums and with public code repositories. This results in greater input from the community as any user can easily submit bug reports or feature suggestions, get involved in discussions on the future direction of Open Babel or even become a developer him/herself. In practice, the number of active contributors has increased over time through this level of open, public development (Figure 3). Moreover, it means that the development of the code is completely transparent and the quality of the software is available for public scrutiny. Indeed, since its inception, over 658 bugs have been submitted to the public tracker and fixed [41].\\n\\nNumber of contributors over time. Note that this graph only includes developers who directly commited code to the Open Babel source code repository, and does not include patches provided by users.\\n\\nValidation and Testing\\n\\nOpen Babel includes an extensive test suite comprising 60 different test programs each with tens to hundreds of tests. In early 2010, a nightly build infrastructure and dashboard was put in place with support from Kitware, Inc. This has greatly improved code quality by catching regressions, and also ensures that the code compiles cleanly on all platforms and compilers supported by Open Babel. Some examples of tests that are run each night are:\\n\\n(1) The MMFF94 forcefield code is tested against the MMFF94 validation suite.\\n\\n(2) The OBAlign class, which was developed using Test-Driven Development (TDD) methodology, is run against its test suite.\\n\\n(3) Handling of symmetry is validated by converting several test cases between SMILES, 2D and 3D SDF, and InChI (there are also several test programs with unit tests for the individual stereo classes in the API).\\n\\n(4) The SMARTS parser is tested using over 250 valid and invalid SMARTS patterns, and the SMARTS matcher is tested using 125 basic SMARTS patterns.\\n\\n(5) The LSSR (Least Set of Smallest Rings) code is tested for invariance against changing the atom order for a series of polycyclic molecules.\\n\\nRecently the development team has placed a major focus on increasing the robustness of file format translation particularly in relation to the commonly used SMILES and MDL Molfile formats. Translating between these formats requires accurate stereochemistry perception, inference of implicit hydrogens, and kekulization of delocalized systems. While it is difficult to ensure that any complex piece of code is free of bugs, and Open Babel is no exception, validation procedures can be carried out to assess the current level of performance and to find additional test cases that expose bugs. The following procedure was used to guide the rewriting of stereochemistry code in Open Babel, a project that began in early 2009. Starting with a dataset of 18,084 3D structures from PubChem3D as an SDF file, we compared the result of (a) conversion to SMILES, followed by conversion of that to Canonical SMILES to (b) conversion directly to Canonical SMILES. This procedure can be used to flush out errors in reading the original SDF file, reading/writing SMILES (either due to stereochemistry errors or kekulization problems), and is also a test (to some extent) of the canonicalization code. At the time of starting this work (March 2009), the error rate found was 1424 (8%); by Oct 2009, combined work on stereochemistry, kekulization and canonicalization had reduced this to 190 (~1%), and continued improvements have reduced the number of errors down to two (shown in Figure 4) for Open Babel 2.3.1 (~0.01%). The first failure is due to a kekulization error in a polycyclic aromatic molecule incorporating heteroatoms: (a) gave c1ccc2c(c1)c1[nH][nH]c3c4c1c(c2)ccc4cc1c3cccc1 while (b) gave c1ccc2c(c1)c1nnc3c4c1c(c2)ccc4cc1c3cccc1. This error led to confusion over whether or not the aromatic nitrogens have hydrogens attached (they do not). The second failure involves confusion over the canonical stereochemistry at a bridgehead carbon: (a) gave C1CN2[C@@H](C1)CCC2 while (b) gave C1CN2[C@H](", "summary": "Open Babel provides several utilities beyond format conversion. One utility is the MMFF94 forcefield code, which is tested against the MMFF94 validation suite. Another utility is the OBAlign class, which was developed using Test-Driven Development (TDD) methodology and is run against its test suite. These utilities demonstrate the software\'s capabilities in molecular modeling and alignment.", "relevance": 10, "cited": true}]}, {"id": "cmms-modalities", "question": "Which two molecular modalities does CMMS-GCL use to extract molecular features?", "answer": "CMMS-GCL utilizes two molecular modalities to extract molecular features: the SMILES sequence and the molecular graph structure (cmms-gcl-2023 chunk 1). The SMILES sequence is processed using a multihead attention BiGRU-based encoder to preserve the context of symbols and learn sequence representations (cmms-gcl-2023 chunk 1). The molecular graph structure is encoded using a graph contrastive learning-based encoder to capture consistencies between local and global structures (cmms-gcl-2023 chunk 1). Additionally, the framework includes an atomic similarity-based sequence encoder that processes SMILES data using a BiGRU model (cmms-gcl-2023 chunk 3). The molecular graph structure encoder in CMMS-GCL incorporates graph contrastive learning to enhance the representation learning of the model by capturing dependencies between local and global structures (cmms-gcl-2023 chunk 6). These two modalities are combined using fully connected neural networks for model training (cmms-gcl-2023 chunk 1). The combination of these two modalities allows for comprehensive feature extraction for metabolic stability prediction (cmms-gcl-2023 chunk 3). The use of both compound SMILES sequences and molecular structures enables CMMS-GCL to learn more informative representations compared to baseline methods that use single modality data (cmms-gcl-2023 chunk 6). The integration of these modalities is central to the model\'s ability to predict metabolic stability effectively.", "references": "1. (cmms-gcl-2023 chunk 1): Du B-X, Long Y, Li X, Wu M, Shi J-Y. CMMS-GCL: cross-modality metabolic stability prediction with graph contrastive learning. Bioinformatics. 2023;39(8):btad503. doi:10.1093/bioinformatics/btad503.\\n\\n2. (cmms-gcl-2023 chunk 3): Du B-X, Long Y, Li X, Wu M, Shi J-Y. CMMS-GCL: cross-modality metabolic stability prediction with graph contrastive learning. Bioinformatics. 2023;39(8):btad503. doi:10.1093/bioinformatics/btad503.\\n\\n3. (cmms-gcl-2023 chunk 6): Du B-X, Long Y, Li X, Wu M, Shi J-Y. CMMS-GCL: cross-modality metabolic stability prediction with graph contrastive learning. Bioinformatics. 2023;39(8):btad503. doi:10.1093/bioinformatics/btad503.", "elapsed_seconds": 19.211, "contexts": [{"document_id": "cmms-gcl-2023", "chunk_name": "cmms-gcl-2023 chunk 3", "source_text": " on the learned representations. Next, we will introduce each module in detail.\\n\\nThe overall framework of CMMS-GCL for metabolic stability prediction. CMMS-GCL is a cross-modality fusion framework augmented via graph contrastive learning, which contains an atomic similarity-based sequence encoder, a molecular graph structure encoder, an inter-view graph contrastive learning, and a stability predictor. Specifically, the atomic similarity-based sequence encoder uses a multihead BiGRU to generate molecular sequence embeddings, where the initial features are obtained by a pretrained Sim2Vec. The molecule graph structure encoder generates molecule embeddings by two-layer GINs. The inter-view graph contrastive learning module augments the original molecular graph by substructure sampling and then contrasts the augmented molecular graph embeddings and the original molecular graph embeddings. Finally, the sequence embeddings and graph embeddings are concatenated and fed into the final predictor for metabolic stability prediction.\\n\\n2.3 Atomic similarity-based sequence encoder\\n\\nCompounds are usually represented by the SMILES (Simplified Molecular-Input Line-Entry System). The local chemical context of an atom reveals its functionality in the SMILES sequence. To capture the context information, we first learn the preliminary features of atoms from SMILES sequence data using Smi2Vec (Lin et al. 2020a,b), which is shown to be capable of preserving the proximity of semantically similar mappings in the embedding space. Following that, we further design an atomic similarity-based sequence encoder to learn representations of compounds. Specifically, the Smi2Vec algorithm first divides a given SMILES string into individual atom or symbol and then looks up the embedding for each atom from a pretrained embedding dictionary. If an atom is not in the dictionary, a random embedding is generated. With the output of Smi2Vec, we can derive the pretrained atom representations Z∈Rm×d1, where each row represents the pretrained representation of an atom. m and d1 denote the number of atoms and the dimension of representations, respectively.\\n\\nAfter deriving the preliminary atom representations, we further design a BiGRU-based encoder to learn representations for compounds. GRU aims to enforce every recurrent unit to preserve the dependencies of different time scales adaptively. Therefore, as a variant of GRU, BiGRU allows the encoder to capture the local chemical context of atoms. Mathematically, the hidden state ht of the t-th atom in the sequence can be formulated as follows: where GRU→(⋅) and GRU→(⋅) represent the forward and backward GRUs, respectively, which captures the interdependence between physically adjacent atoms within the sequence. zt is the preliminary representation vector of the t-th atom in the sequence. zt−1→ and zt+1→ denote the hidden states of the (t−1)-th and (t+1)-th atoms from the forward and backward GRUs, respectively. As the output of the BiGRU model, we can generate the updated atom representation H∈Rm×d2 with dimension d2.\\n\\nPrevious study shows that key functional groups (i.e. molecule substructure) play a critical role in metabolic stability (Li et al. 2022). To effectively identify key functional groups, we introduce an attention mechanism (Zheng et al. 2020) to the BiGRU model. In particular, given a molecule i, we first learn the importance of all atoms by the following attention coefficient: where αij∈R1×m is attention coefficient that denotes the importance of atom j to the molecule i. MLP(⋅) is a trainable feedforward neural network with an activation function (i.e. tanh). Softmax[⋅] is a normalization function that ensures the sum of attention scores of all atoms is equal to 1. Subsequently, we derive the molecule representation si by aggregating the representation of all the atoms according to their attention coefficients: where FCs(⋅) is a fully connected neural network.\\n\\n2.4 Molecular graph structure encoder\\n\\nIn this section, we introduce the graph isomorphism network (GIN) to learn the representation of molecules from molecular structures. Given a molecule i, let us denote its graph structure as G={V,E}, where V is a set of N nodes (i.e. atoms) and E is a set of edges (i.e. chemical bonds connecting atoms). We denote A∈RN×N as the adjacency matrix of the graph, in which avu=1 indicates atoms v and u are connected by a bond while avu=0 indicates atoms v and u are disconnected by a bond. N is the number of atoms. The initial input features of the molecule i are defined by several atom-level properties such as the atom symbol, the number of adjacent atoms, the number of adjacent hydrogens, the implicit value of the atom, and the atom occurrence in an aromatic structure.\\n\\nFor each atom v in the molecular graph G, GIN learns its representation by iteratively aggregating the representations of its neighbors. Specifically, the representation hv(k) for atom v in the kth layer is defined as follows: where ϵ(k) is a learnable weight parameter. hv(k) is the representation of node v at the k-th iteration. MLP(⋅) is a feedforward neural network. hv(k−1) is the representation of node v at the (k−1)-th iteration. N(v) denotes the set of neighbors of the node v. With K iterations, the representation hv can effectively capture K-hop neighborhood information. Eventually, we take the output of the K-th iteration as the final representation hv for the atom v. To obtain the molecule representation pi∈R1×", "summary": "CMMS-GCL is a cross-modality fusion framework that uses two molecular modalities to extract molecular features. These modalities are the molecular sequence data represented by SMILES strings and the molecular graph structure. The framework includes an atomic similarity-based sequence encoder that processes SMILES data using a BiGRU model and a molecular graph structure encoder that uses GINs to learn representations from molecular graphs. The combination of these two modalities allows for comprehensive feature extraction for metabolic stability prediction.", "relevance": 10, "cited": true}, {"document_id": "cmms-gcl-2023", "chunk_name": "cmms-gcl-2023 chunk 6", "source_text": " the feed-forward neural network are the same as those in CMMS-GCL.\\n\\nAttentiveFP (Xiong et al. 2020): it implemented the same node features of CMMS-GCL and extra 10-dim bond features as input. Then it used a three-layer message-passing neural network and two-time steps in the message-passing process.\\n\\nThe first five approaches utilized the parameter settings recommended by the original reference literature, while for the latter two methods, we identified the optimal settings.\\n\\n3.3 Performance evaluation on HLM dataset\\n\\nTable 2 shows the results of 10-fold cross-validation on HLM dataset. Among all the methods, CMMS-GCL achieves the best performance with an average AUC of 0.866±0.013, average Accuracy of 0.805±0.016, average F1 score of 0.853±0.011, and average MCC of 0.570±0.034, which are 1.05%, 1.00%, 0.59%, and 1.42% higher than that of the second-best methods. The results demonstrate the effectiveness of CMMS-GCL in predicting the metabolic stability of compound candidates.\\n\\nPerformance evaluation of CMMS-GCL with baseline methods on HLM metabolic stability prediction.\\n\\nThe best results are marked in bold and the second best is underlined.\\n\\nThere are two main reasons why our model outperforms baselines. On the one hand, CMMS-GCL combines both compound SMILES sequences with molecular structures, which allows our model to learn more informative representations than baseline methods that use single modality data (i.e. Ryu et al. 2022). On the other hand, in the molecular graph structure encoder, we introduce graph contrastive learning to capture the dependencies between local and global structures, which enhances the representation learning of our model. We will demonstrate it in the following ablation study.\\n\\n3.4 Independent evaluation on the external dataset\\n\\nTo further validate the effectiveness and generalizability of our model, we carried out CMMS-GCL and baseline methods on an external dataset. Note that all the methods utilize the HLM dataset for model training and use the external data as a test dataset.\\n\\nThe results in Table 3 show that CMMS-GCL achieved the best performance compared with seven methods across all metrics on 10 times average results. More specifically, CMMS-GCL surpasses the second-best method by 6.60%, 8.17%, 4.45%, and 27.31% in terms of AUC, Accuracy, F1 score, and MCC, respectively. Furthermore, we derived a subset of molecules from the external dataset, which shows low similarity to the HLM dataset. We also compared our CMMS-GCL with various baselines on this subset of molecules with low structural similarity (<0.5) as shown in Table 4. We can observe the improvement of our method upon the second-best by 7.72%, 6.31%, 3.19%, and 38.60% in terms of AUC, Accuracy, F1 score, and MCC, respectively. More importantly, compared with the results on the HLM dataset (Table 2), CMMS-GCL exhibited a more substantial improvement on the external dataset (Tables 3 and 4). In contrast, for baseline methods, especially methods that are based on traditional machine learning, their performance decrease to varying degrees. Therefore, we can make a conclusion that the proposed CMMS-GCL is an effective and robust computational model for predicting the metabolic stability of compounds.\\n\\nPerformance evaluation of CMMS-GCL with baseline methods on the external datasets.\\n\\nThe best results are marked in bold and the second best is underlined.\\n\\nPerformance evaluation of CMMS-GCL against baseline methods using a subset of external datasets with low molecular similarity.\\n\\nThe best results are marked in bold and the second best is underlined.\\n\\n3.5 Ablation studies\\n\\nOur CMMS-GCL consists of an atom similarity-based sequence encoder, a molecular graph structure encoder, an inter-view graph contrastive learning, and a stability predictor, as shown in Fig. 1. In this section, we perform an ablation study on two datasets to evaluate the contributions of the cross-modality encoder and the inter-view graph contrastive learning. Note that in Tables 5 and 6, “w/o CM” denotes the variant of CMMS-GCL only reserves molecular graph structure encoder and stability predictor. “w/o GCL” denotes the variant of CMMS-GCL without inter-view graph contrastive learning. Tables 5 and 6 report the performance comparison between CMMS-GCL and its variants on HLM and the external dataset in terms of four metrics, respectively. It can be discovered that both cross-modality encoder and inter-view graph contrastive learning contribute to the performance improvement of the model. Specifically, in Table 5, compared with the variants w/o CM and w/o GCL, CMMS-GCL improves on AUC by 0.81% and 0.58%, on Accuracy by 1.13% and 0.75%, on F1 score by 1.55% and 0.95%, on MCC by 1.60% and 0.88%, respectively. Moreover, on the external dataset, compared with the variants w/o CM and w/o GCL, CMMS-GCL improves on AUC by 2.19% and 1.49%, on Accuracy by 2.46% and 1.46%, on F1 score by 1.48% and 0.56%, on MCC by 14.40% and 12.", "summary": "CMMS-GCL uses both compound SMILES sequences and molecular structures to extract molecular features. The model combines these two modalities to learn more informative representations compared to baseline methods that use single modality data. Additionally, the molecular graph structure encoder in CMMS-GCL incorporates graph contrastive learning to capture dependencies between local and global structures, enhancing the representation learning of the model.", "relevance": 10, "cited": true}, {"document_id": "cmms-gcl-2023", "chunk_name": "cmms-gcl-2023 chunk 7", "source_text": "55% and 0.95%, on MCC by 1.60% and 0.88%, respectively. Moreover, on the external dataset, compared with the variants w/o CM and w/o GCL, CMMS-GCL improves on AUC by 2.19% and 1.49%, on Accuracy by 2.46% and 1.46%, on F1 score by 1.48% and 0.56%, on MCC by 14.40% and 12.13%, respectively.\\n\\nAblation studies of CMMS-GCL, which include two components variants and three augmentation methods variants.\\n\\nThe best results are marked in bold.\\n\\nAblation studies of CMMS-GCL on external dataset, which include two components variants and three augmentation methods variants.\\n\\nThe best results are marked in bold.\\n\\nMoreover, to further validate the superiority of the substructure sampling-based graph augmentation method, we compared it with other three augmentation methods: Node Dropping, Edge Perturbation, and Attribute Masking. The sampling ratios, which determine the fraction of nodes/edges/features selected by these three augmentations, were set to 0.2, 0.4, and 0.4, respectively. The results in Tables 5 and 6 show that the substructure sampling-based method achieves the best performance consistently and improves the second-best method by 0.35%, 0.37%, 0.35%, and 1.24% in terms of AUC, accuracy, F1 score, and MCC, respectively, in Table 5 and improves the second-best method by 1.60%, 3.09%, 1.60%, and 20.51% in terms of AUC, accuracy, F1 score, and MCC, respectively, in Table 6. The results indicate that the substructure sampling-based graph contrastive learning plays a vital role in predicting metabolic stability.\\n\\n3.6 Performance evaluation on novel structural molecules\\n\\nTo validate our CMMS-GCL’s capacity for novel molecular structures, we calculated the Tanimoto similarity among the molecules within the HLM dataset using their ECFP fingerprints. We then performed a K-means clustering analysis on the similarity matrix, resulting in five clear clusters. To present our results more tangibly, we employed Principal Component Analysis (PCA) for dimensionality reduction, as depicted in Fig. 3. This dimension reduction allowed us to visualize and interpret the clustering results more effectively, thereby accentuating the efficiency of our CMMS-GCL in identifying distinct molecular structures.\\n\\nPCA visualization of K-means clustering on the HLM dataset, where the values in parentheses correspond to the explained variance ratio.\\n\\nFurthermore, we conducted the leave-one-cluster-out cross-validation on the identified clusters, providing a robust validation of our approach on the molecules with novel structures. We compared our CMMS-GCL against various baseline methods under this leave-one-cluster-out setting in Fig. 4. We can observe that CMMS-GCL outperforms the second-best method by 1.99%, 0.24%, 0.36%, and 3.82% in terms of AUC, Accuracy, F1 score, and MCC, respectively.\\n\\nPerformance comparison of CMMS-GCL and baseline methods on HLM dataset based on leave-one-cluster-out strategy.\\n\\nIn conclusion, the assessment of structurally unique compounds manifests the superior predictive performance of CMMS-GCL, underscoring its competency in handling diverse chemical structures. This revelation accentuates the potential of CMMS-GCL as an invaluable asset for medicinal chemists in pinpointing compounds with novel structures and predicting their respective activities.\\n\\n3.7 Case study on interpretability of CMMS-GCL\\n\\nAlthough deep learning is often known as a black-box model, it is crucial to comprehend the prediction mechanism employed by CMMS-GCL for distinguishing stable and unstable molecules, as well as to ascertain its potential influence on lead compound optimization. To thoroughly investigate the interpretability of our model, we embarked on an in-depth examination of its two primary modules, focusing distinctly on the sequence-based and the graph structure-based components.\\n\\n3.7.1 Sequence-based interpretability\\n\\nSince the atomic similarity-based sequence encoder module can capture the molecule’s important functional groups through its multihead attention module, we determined that two bonding atoms are regarded as a crucial functional group. The weight of a bond is the average of the weights of its constituent atoms (partial bonds that can be visualized are also averaged together of SMILES symbols) and are highlighted in Fig. 5.\\n\\nInterpretability case study of crucial functional groups for stable and unstable molecules based on sequences. (a) Four stable molecules with high metabolic stability. (b) Four unstable molecules with low metabolic stability. The atoms and bonds of high/low metabolic stability-specific functional groups are highlighted.\\n\\nWe selected four molecules exhibiting high metabolic stability. The highlighted regions reveal specific areas within the molecules that contribute to metabolic stability or instability. As shown in Fig. 5a, for the first two stable molecules, their highlighted regions exhibit acetylene functional groups, which are commonly associated with enhanced stability and are less prone to metabolic reactions (Talele 2020). For the other two molecules, the trifluoromethyl group exhibits strong electronegativity and chemical inertness, making it difficult to undergo a transformation during metabolic processes in biological systems (Liang et al. 2013).\\n\\nIn contrast, another four molecules display low metabolic stability (i.e. instability). For these unstable molecules prone to metabolism, their prominent functional groups involve various metabolic pathways. As shown in Fig. 5b, the attention scores indicate that specific functional groups, such as amide, thiophene,", "summary": "The CMMS-GCL method utilizes cross-modality approaches to extract molecular features, integrating both sequence-based and graph structure-based components. The sequence-based component captures molecular features through a multihead attention module, focusing on functional groups like acetylene and trifluoromethyl. The graph structure-based component employs substructure sampling-based graph contrastive learning to enhance the prediction of metabolic stability. These two modalities work together to improve the model\'s performance in predicting metabolic stability.", "relevance": 10, "cited": false}, {"document_id": "cmms-gcl-2023", "chunk_name": "cmms-gcl-2023 chunk 1", "source_text": "CMMS-GCL: cross-modality metabolic stability prediction with graph contrastive learning\\n\\nDu B-X, Long Y, Li X, Wu M, Shi J-Y. CMMS-GCL: cross-modality metabolic stability prediction with graph contrastive learning. Bioinformatics. 2023;39(8):btad503. doi:10.1093/bioinformatics/btad503.\\n\\nhttps://creativecommons.org/licenses/by/4.0/\\n\\nAbstract\\n\\nMotivation\\n\\nMetabolic stability plays a crucial role in the early stages of drug discovery and development. Accurately modeling and predicting molecular metabolic stability has great potential for the efficient screening of drug candidates as well as the optimization of lead compounds. Considering wet-lab experiment is time-consuming, laborious, and expensive, in silico prediction of metabolic stability is an alternative choice. However, few computational methods have been developed to address this task. In addition, it remains a significant challenge to explain key functional groups determining metabolic stability.\\n\\nResults\\n\\nTo address these issues, we develop a novel cross-modality graph contrastive learning model named CMMS-GCL for predicting the metabolic stability of drug candidates. In our framework, we design deep learning methods to extract features for molecules from two modality data, i.e. SMILES sequence and molecule graph. In particular, for the sequence data, we design a multihead attention BiGRU-based encoder to preserve the context of symbols to learn sequence representations of molecules. For the graph data, we propose a graph contrastive learning-based encoder to learn structure representations by effectively capturing the consistencies between local and global structures. We further exploit fully connected neural networks to combine the sequence and structure representations for model training. Extensive experimental results on two datasets demonstrate that our CMMS-GCL consistently outperforms seven state-of-the-art methods. Furthermore, a collection of case studies on sequence data and statistical analyses of the graph structure module strengthens the validation of the interpretability of crucial functional groups recognized by CMMS-GCL. Overall, CMMS-GCL can serve as an effective and interpretable tool for predicting metabolic stability, identifying critical functional groups, and thus facilitating the drug discovery process and lead compound optimization.\\n\\nAvailability and implementation\\n\\nThe code and data underlying this article are freely available at https://github.com/dubingxue/CMMS-GCL.\\n\\n1 Introduction\\n\\nMetabolic stability, which refers to the extent and rate of drug metabolism in the body, plays a crucial role in the early stages of drug discovery and development (Pritchard et al. 2003). It significantly impacts various pharmacokinetic (PK) processes, such as oral bioavailability, the volume of distribution, clearance, half-life, and toxicity, ultimately influencing the appropriate drug dose and frequency (Davies et al. 2020, Gajula et al. 2021). Therefore, improving the metabolic stability of hit/lead compounds in the early phases of drug discovery has a pivotal role w.r.t. drug candidate screening and lead compound optimization (Kirchmair et al. 2015).\\n\\nMost of the early metabolic stability assays are conducted in vitro by incubating hit/lead compounds with liver microsomes (LMs) to assess their metabolic properties (Gajula et al. 2021). Notably, LMs are abundant in subcellular components containing cytochrome P450 (CYP450) enzymes, which play a crucial role in drug metabolism, and their assays provide preliminary guidance for more definitive PK assessments (Słoczyńska et al. 2019). Furthermore, in vitro mouse liver microsomes (MLMs), rat liver microsomes (RLMs), and human liver microsomes (HLMs) have emerged as essential approaches for initial assessment of metabolic stability (Mak et al. 2022, Rodríguez-Pérez et al. 2023). However, experimental determination of molecular metabolic stability is time-consuming, laborious, and expensive (Sodhi and Benet 2021). Therefore, there is an urgent need to develop computational methods to assist experimental screenings of metabolic stability.\\n\\nIn the past several years, traditional machine learning-based methods have been developed for metabolic stability prediction. For example, Perryman et al. (2016) first collected a dataset of MLM compounds from PubChem and presented a Bayesian model to predict molecular metabolic stability using nine molecular descriptors (e.g. fingerprints). Podlewska and Kafel (2018) built an online platform called MetStabOn for metabolic stability prediction, where multiple kinds of traditional machine learning methods were assembled, such as k-nearest neighbor (kNN), Naïve Bayes (NB), and random forest (RF). MetStabOn takes PaDEL-descriptors and extended fingerprints as input and can be applied for MLMs, RLMs, and HLMs data. Recently, Ryu et al. (2022) first generated an in-house HLMs dataset and introduced a RF model to evaluate metabolic stability with molecular descriptors as inputs. However, these methods take fingerprint or descriptor information into account only, while ignoring the molecule structural information, which results in suboptimal performance.\\n\\nThe rapid development of machine learning has led to the widespread application of graph neural networks (GNNs) in various biological research fields, such as drug–target interaction prediction (Long et al. 2022, Wang et al. 2022), molecular property prediction (W", "summary": "CMMS-GCL is a novel cross-modality graph contrastive learning model designed for predicting the metabolic stability of drug candidates. The model utilizes two molecular modalities to extract molecular features: SMILES sequence and molecule graph. For the SMILES sequence, a multihead attention BiGRU-based encoder is used to preserve the context of symbols and learn sequence representations. For the molecule graph, a graph contrastive learning-based encoder is employed to capture the consistencies between local and global structures. These features are then combined using fully connected neural networks for model training.", "relevance": 10, "cited": true}]}, {"id": "cmms-task", "question": "What prediction task was CMMS-GCL developed to address?", "answer": "CMMS-GCL was developed to address the prediction task of metabolic stability for drug candidates (cmms-gcl-2023 chunk 1). Metabolic stability is crucial in the early stages of drug discovery and development, as it impacts pharmacokinetic processes and drug efficacy (cmms-gcl-2023 chunk 1). The model integrates cross-modality data, including SMILES sequences and molecule graphs, and employs graph contrastive learning to capture structural and sequence representations (cmms-gcl-2023 chunk 1). It is a cross-modality fusion framework that integrates sequence and molecular graph data using graph contrastive learning (cmms-gcl-2023 chunk 3). The framework includes an atomic similarity-based sequence encoder, a molecular graph structure encoder, an inter-view graph contrastive learning module, and a stability predictor (cmms-gcl-2023 chunk 3). The sequence encoder uses a BiGRU model with attention mechanisms to capture local chemical context, while the graph encoder employs GINs to learn molecular structure representations (cmms-gcl-2023 chunk 3). The model combines both compound SMILES sequences with molecular structures to learn more informative representations (cmms-gcl-2023 chunk 6). It also introduces graph contrastive learning to capture dependencies between local and global structures, enhancing representation learning (cmms-gcl-2023 chunk 6). The model was evaluated on the HLM dataset and an external dataset, achieving the best performance compared to baseline methods in terms of AUC, Accuracy, F1 score, and MCC (cmms-gcl-2023 chunk 6).", "references": "1. (cmms-gcl-2023 chunk 1): Du B-X, Long Y, Li X, Wu M, Shi J-Y. CMMS-GCL: cross-modality metabolic stability prediction with graph contrastive learning. Bioinformatics. 2023;39(8):btad503. doi:10.1093/bioinformatics/btad503.\\n\\n2. (cmms-gcl-2023 chunk 3): Du B-X, Long Y, Li X, Wu M, Shi J-Y. CMMS-GCL: cross-modality metabolic stability prediction with graph contrastive learning. Bioinformatics. 2023;39(8):btad503. doi:10.1093/bioinformatics/btad503.\\n\\n3. (cmms-gcl-2023 chunk 6): Du B-X, Long Y, Li X, Wu M, Shi J-Y. CMMS-GCL: cross-modality metabolic stability prediction with graph contrastive learning. Bioinformatics. 2023;39(8):btad503. doi:10.1093/bioinformatics/btad503.", "elapsed_seconds": 20.745, "contexts": [{"document_id": "cmms-gcl-2023", "chunk_name": "cmms-gcl-2023 chunk 1", "source_text": "CMMS-GCL: cross-modality metabolic stability prediction with graph contrastive learning\\n\\nDu B-X, Long Y, Li X, Wu M, Shi J-Y. CMMS-GCL: cross-modality metabolic stability prediction with graph contrastive learning. Bioinformatics. 2023;39(8):btad503. doi:10.1093/bioinformatics/btad503.\\n\\nhttps://creativecommons.org/licenses/by/4.0/\\n\\nAbstract\\n\\nMotivation\\n\\nMetabolic stability plays a crucial role in the early stages of drug discovery and development. Accurately modeling and predicting molecular metabolic stability has great potential for the efficient screening of drug candidates as well as the optimization of lead compounds. Considering wet-lab experiment is time-consuming, laborious, and expensive, in silico prediction of metabolic stability is an alternative choice. However, few computational methods have been developed to address this task. In addition, it remains a significant challenge to explain key functional groups determining metabolic stability.\\n\\nResults\\n\\nTo address these issues, we develop a novel cross-modality graph contrastive learning model named CMMS-GCL for predicting the metabolic stability of drug candidates. In our framework, we design deep learning methods to extract features for molecules from two modality data, i.e. SMILES sequence and molecule graph. In particular, for the sequence data, we design a multihead attention BiGRU-based encoder to preserve the context of symbols to learn sequence representations of molecules. For the graph data, we propose a graph contrastive learning-based encoder to learn structure representations by effectively capturing the consistencies between local and global structures. We further exploit fully connected neural networks to combine the sequence and structure representations for model training. Extensive experimental results on two datasets demonstrate that our CMMS-GCL consistently outperforms seven state-of-the-art methods. Furthermore, a collection of case studies on sequence data and statistical analyses of the graph structure module strengthens the validation of the interpretability of crucial functional groups recognized by CMMS-GCL. Overall, CMMS-GCL can serve as an effective and interpretable tool for predicting metabolic stability, identifying critical functional groups, and thus facilitating the drug discovery process and lead compound optimization.\\n\\nAvailability and implementation\\n\\nThe code and data underlying this article are freely available at https://github.com/dubingxue/CMMS-GCL.\\n\\n1 Introduction\\n\\nMetabolic stability, which refers to the extent and rate of drug metabolism in the body, plays a crucial role in the early stages of drug discovery and development (Pritchard et al. 2003). It significantly impacts various pharmacokinetic (PK) processes, such as oral bioavailability, the volume of distribution, clearance, half-life, and toxicity, ultimately influencing the appropriate drug dose and frequency (Davies et al. 2020, Gajula et al. 2021). Therefore, improving the metabolic stability of hit/lead compounds in the early phases of drug discovery has a pivotal role w.r.t. drug candidate screening and lead compound optimization (Kirchmair et al. 2015).\\n\\nMost of the early metabolic stability assays are conducted in vitro by incubating hit/lead compounds with liver microsomes (LMs) to assess their metabolic properties (Gajula et al. 2021). Notably, LMs are abundant in subcellular components containing cytochrome P450 (CYP450) enzymes, which play a crucial role in drug metabolism, and their assays provide preliminary guidance for more definitive PK assessments (Słoczyńska et al. 2019). Furthermore, in vitro mouse liver microsomes (MLMs), rat liver microsomes (RLMs), and human liver microsomes (HLMs) have emerged as essential approaches for initial assessment of metabolic stability (Mak et al. 2022, Rodríguez-Pérez et al. 2023). However, experimental determination of molecular metabolic stability is time-consuming, laborious, and expensive (Sodhi and Benet 2021). Therefore, there is an urgent need to develop computational methods to assist experimental screenings of metabolic stability.\\n\\nIn the past several years, traditional machine learning-based methods have been developed for metabolic stability prediction. For example, Perryman et al. (2016) first collected a dataset of MLM compounds from PubChem and presented a Bayesian model to predict molecular metabolic stability using nine molecular descriptors (e.g. fingerprints). Podlewska and Kafel (2018) built an online platform called MetStabOn for metabolic stability prediction, where multiple kinds of traditional machine learning methods were assembled, such as k-nearest neighbor (kNN), Naïve Bayes (NB), and random forest (RF). MetStabOn takes PaDEL-descriptors and extended fingerprints as input and can be applied for MLMs, RLMs, and HLMs data. Recently, Ryu et al. (2022) first generated an in-house HLMs dataset and introduced a RF model to evaluate metabolic stability with molecular descriptors as inputs. However, these methods take fingerprint or descriptor information into account only, while ignoring the molecule structural information, which results in suboptimal performance.\\n\\nThe rapid development of machine learning has led to the widespread application of graph neural networks (GNNs) in various biological research fields, such as drug–target interaction prediction (Long et al. 2022, Wang et al. 2022), molecular property prediction (W", "summary": "CMMS-GCL was developed to address the prediction task of metabolic stability for drug candidates. Metabolic stability is crucial in the early stages of drug discovery and development, as it impacts pharmacokinetic processes and drug efficacy. The model uses cross-modality data, including SMILES sequences and molecule graphs, and employs graph contrastive learning to capture structural and sequence representations. It outperforms existing methods and provides interpretable insights into critical functional groups affecting metabolic stability.", "relevance": 10, "cited": true}, {"document_id": "cmms-gcl-2023", "chunk_name": "cmms-gcl-2023 chunk 3", "source_text": " on the learned representations. Next, we will introduce each module in detail.\\n\\nThe overall framework of CMMS-GCL for metabolic stability prediction. CMMS-GCL is a cross-modality fusion framework augmented via graph contrastive learning, which contains an atomic similarity-based sequence encoder, a molecular graph structure encoder, an inter-view graph contrastive learning, and a stability predictor. Specifically, the atomic similarity-based sequence encoder uses a multihead BiGRU to generate molecular sequence embeddings, where the initial features are obtained by a pretrained Sim2Vec. The molecule graph structure encoder generates molecule embeddings by two-layer GINs. The inter-view graph contrastive learning module augments the original molecular graph by substructure sampling and then contrasts the augmented molecular graph embeddings and the original molecular graph embeddings. Finally, the sequence embeddings and graph embeddings are concatenated and fed into the final predictor for metabolic stability prediction.\\n\\n2.3 Atomic similarity-based sequence encoder\\n\\nCompounds are usually represented by the SMILES (Simplified Molecular-Input Line-Entry System). The local chemical context of an atom reveals its functionality in the SMILES sequence. To capture the context information, we first learn the preliminary features of atoms from SMILES sequence data using Smi2Vec (Lin et al. 2020a,b), which is shown to be capable of preserving the proximity of semantically similar mappings in the embedding space. Following that, we further design an atomic similarity-based sequence encoder to learn representations of compounds. Specifically, the Smi2Vec algorithm first divides a given SMILES string into individual atom or symbol and then looks up the embedding for each atom from a pretrained embedding dictionary. If an atom is not in the dictionary, a random embedding is generated. With the output of Smi2Vec, we can derive the pretrained atom representations Z∈Rm×d1, where each row represents the pretrained representation of an atom. m and d1 denote the number of atoms and the dimension of representations, respectively.\\n\\nAfter deriving the preliminary atom representations, we further design a BiGRU-based encoder to learn representations for compounds. GRU aims to enforce every recurrent unit to preserve the dependencies of different time scales adaptively. Therefore, as a variant of GRU, BiGRU allows the encoder to capture the local chemical context of atoms. Mathematically, the hidden state ht of the t-th atom in the sequence can be formulated as follows: where GRU→(⋅) and GRU→(⋅) represent the forward and backward GRUs, respectively, which captures the interdependence between physically adjacent atoms within the sequence. zt is the preliminary representation vector of the t-th atom in the sequence. zt−1→ and zt+1→ denote the hidden states of the (t−1)-th and (t+1)-th atoms from the forward and backward GRUs, respectively. As the output of the BiGRU model, we can generate the updated atom representation H∈Rm×d2 with dimension d2.\\n\\nPrevious study shows that key functional groups (i.e. molecule substructure) play a critical role in metabolic stability (Li et al. 2022). To effectively identify key functional groups, we introduce an attention mechanism (Zheng et al. 2020) to the BiGRU model. In particular, given a molecule i, we first learn the importance of all atoms by the following attention coefficient: where αij∈R1×m is attention coefficient that denotes the importance of atom j to the molecule i. MLP(⋅) is a trainable feedforward neural network with an activation function (i.e. tanh). Softmax[⋅] is a normalization function that ensures the sum of attention scores of all atoms is equal to 1. Subsequently, we derive the molecule representation si by aggregating the representation of all the atoms according to their attention coefficients: where FCs(⋅) is a fully connected neural network.\\n\\n2.4 Molecular graph structure encoder\\n\\nIn this section, we introduce the graph isomorphism network (GIN) to learn the representation of molecules from molecular structures. Given a molecule i, let us denote its graph structure as G={V,E}, where V is a set of N nodes (i.e. atoms) and E is a set of edges (i.e. chemical bonds connecting atoms). We denote A∈RN×N as the adjacency matrix of the graph, in which avu=1 indicates atoms v and u are connected by a bond while avu=0 indicates atoms v and u are disconnected by a bond. N is the number of atoms. The initial input features of the molecule i are defined by several atom-level properties such as the atom symbol, the number of adjacent atoms, the number of adjacent hydrogens, the implicit value of the atom, and the atom occurrence in an aromatic structure.\\n\\nFor each atom v in the molecular graph G, GIN learns its representation by iteratively aggregating the representations of its neighbors. Specifically, the representation hv(k) for atom v in the kth layer is defined as follows: where ϵ(k) is a learnable weight parameter. hv(k) is the representation of node v at the k-th iteration. MLP(⋅) is a feedforward neural network. hv(k−1) is the representation of node v at the (k−1)-th iteration. N(v) denotes the set of neighbors of the node v. With K iterations, the representation hv can effectively capture K-hop neighborhood information. Eventually, we take the output of the K-th iteration as the final representation hv for the atom v. To obtain the molecule representation pi∈R1×", "summary": "CMMS-GCL was developed to address the prediction task of metabolic stability. It is a cross-modality fusion framework that integrates sequence and molecular graph data using graph contrastive learning. The framework includes an atomic similarity-based sequence encoder, a molecular graph structure encoder, an inter-view graph contrastive learning module, and a stability predictor. The sequence encoder uses a BiGRU model with attention mechanisms to capture local chemical context, while the graph encoder employs GINs to learn molecular structure representations. The final predictor combines these features to predict metabolic stability.", "relevance": 10, "cited": true}, {"document_id": "cmms-gcl-2023", "chunk_name": "cmms-gcl-2023 chunk 7", "source_text": "55% and 0.95%, on MCC by 1.60% and 0.88%, respectively. Moreover, on the external dataset, compared with the variants w/o CM and w/o GCL, CMMS-GCL improves on AUC by 2.19% and 1.49%, on Accuracy by 2.46% and 1.46%, on F1 score by 1.48% and 0.56%, on MCC by 14.40% and 12.13%, respectively.\\n\\nAblation studies of CMMS-GCL, which include two components variants and three augmentation methods variants.\\n\\nThe best results are marked in bold.\\n\\nAblation studies of CMMS-GCL on external dataset, which include two components variants and three augmentation methods variants.\\n\\nThe best results are marked in bold.\\n\\nMoreover, to further validate the superiority of the substructure sampling-based graph augmentation method, we compared it with other three augmentation methods: Node Dropping, Edge Perturbation, and Attribute Masking. The sampling ratios, which determine the fraction of nodes/edges/features selected by these three augmentations, were set to 0.2, 0.4, and 0.4, respectively. The results in Tables 5 and 6 show that the substructure sampling-based method achieves the best performance consistently and improves the second-best method by 0.35%, 0.37%, 0.35%, and 1.24% in terms of AUC, accuracy, F1 score, and MCC, respectively, in Table 5 and improves the second-best method by 1.60%, 3.09%, 1.60%, and 20.51% in terms of AUC, accuracy, F1 score, and MCC, respectively, in Table 6. The results indicate that the substructure sampling-based graph contrastive learning plays a vital role in predicting metabolic stability.\\n\\n3.6 Performance evaluation on novel structural molecules\\n\\nTo validate our CMMS-GCL’s capacity for novel molecular structures, we calculated the Tanimoto similarity among the molecules within the HLM dataset using their ECFP fingerprints. We then performed a K-means clustering analysis on the similarity matrix, resulting in five clear clusters. To present our results more tangibly, we employed Principal Component Analysis (PCA) for dimensionality reduction, as depicted in Fig. 3. This dimension reduction allowed us to visualize and interpret the clustering results more effectively, thereby accentuating the efficiency of our CMMS-GCL in identifying distinct molecular structures.\\n\\nPCA visualization of K-means clustering on the HLM dataset, where the values in parentheses correspond to the explained variance ratio.\\n\\nFurthermore, we conducted the leave-one-cluster-out cross-validation on the identified clusters, providing a robust validation of our approach on the molecules with novel structures. We compared our CMMS-GCL against various baseline methods under this leave-one-cluster-out setting in Fig. 4. We can observe that CMMS-GCL outperforms the second-best method by 1.99%, 0.24%, 0.36%, and 3.82% in terms of AUC, Accuracy, F1 score, and MCC, respectively.\\n\\nPerformance comparison of CMMS-GCL and baseline methods on HLM dataset based on leave-one-cluster-out strategy.\\n\\nIn conclusion, the assessment of structurally unique compounds manifests the superior predictive performance of CMMS-GCL, underscoring its competency in handling diverse chemical structures. This revelation accentuates the potential of CMMS-GCL as an invaluable asset for medicinal chemists in pinpointing compounds with novel structures and predicting their respective activities.\\n\\n3.7 Case study on interpretability of CMMS-GCL\\n\\nAlthough deep learning is often known as a black-box model, it is crucial to comprehend the prediction mechanism employed by CMMS-GCL for distinguishing stable and unstable molecules, as well as to ascertain its potential influence on lead compound optimization. To thoroughly investigate the interpretability of our model, we embarked on an in-depth examination of its two primary modules, focusing distinctly on the sequence-based and the graph structure-based components.\\n\\n3.7.1 Sequence-based interpretability\\n\\nSince the atomic similarity-based sequence encoder module can capture the molecule’s important functional groups through its multihead attention module, we determined that two bonding atoms are regarded as a crucial functional group. The weight of a bond is the average of the weights of its constituent atoms (partial bonds that can be visualized are also averaged together of SMILES symbols) and are highlighted in Fig. 5.\\n\\nInterpretability case study of crucial functional groups for stable and unstable molecules based on sequences. (a) Four stable molecules with high metabolic stability. (b) Four unstable molecules with low metabolic stability. The atoms and bonds of high/low metabolic stability-specific functional groups are highlighted.\\n\\nWe selected four molecules exhibiting high metabolic stability. The highlighted regions reveal specific areas within the molecules that contribute to metabolic stability or instability. As shown in Fig. 5a, for the first two stable molecules, their highlighted regions exhibit acetylene functional groups, which are commonly associated with enhanced stability and are less prone to metabolic reactions (Talele 2020). For the other two molecules, the trifluoromethyl group exhibits strong electronegativity and chemical inertness, making it difficult to undergo a transformation during metabolic processes in biological systems (Liang et al. 2013).\\n\\nIn contrast, another four molecules display low metabolic stability (i.e. instability). For these unstable molecules prone to metabolism, their prominent functional groups involve various metabolic pathways. As shown in Fig. 5b, the attention scores indicate that specific functional groups, such as amide, thiophene,", "summary": "CMMS-GCL was developed to address the prediction task of metabolic stability for molecules. The model employs cross-modality metabolic stability prediction with graph contrastive learning. It was evaluated on both internal and external datasets, showing significant improvements in metrics such as AUC, Accuracy, F1 score, and MCC compared to baseline methods. The model\'s performance was further validated on novel structural molecules using K-means clustering and PCA visualization, demonstrating its effectiveness in identifying distinct molecular structures. Additionally, the model\'s interpretability was explored through sequence-based and graph structure-based components, highlighting crucial functional groups for metabolic stability.", "relevance": 10, "cited": false}, {"document_id": "cmms-gcl-2023", "chunk_name": "cmms-gcl-2023 chunk 6", "source_text": " the feed-forward neural network are the same as those in CMMS-GCL.\\n\\nAttentiveFP (Xiong et al. 2020): it implemented the same node features of CMMS-GCL and extra 10-dim bond features as input. Then it used a three-layer message-passing neural network and two-time steps in the message-passing process.\\n\\nThe first five approaches utilized the parameter settings recommended by the original reference literature, while for the latter two methods, we identified the optimal settings.\\n\\n3.3 Performance evaluation on HLM dataset\\n\\nTable 2 shows the results of 10-fold cross-validation on HLM dataset. Among all the methods, CMMS-GCL achieves the best performance with an average AUC of 0.866±0.013, average Accuracy of 0.805±0.016, average F1 score of 0.853±0.011, and average MCC of 0.570±0.034, which are 1.05%, 1.00%, 0.59%, and 1.42% higher than that of the second-best methods. The results demonstrate the effectiveness of CMMS-GCL in predicting the metabolic stability of compound candidates.\\n\\nPerformance evaluation of CMMS-GCL with baseline methods on HLM metabolic stability prediction.\\n\\nThe best results are marked in bold and the second best is underlined.\\n\\nThere are two main reasons why our model outperforms baselines. On the one hand, CMMS-GCL combines both compound SMILES sequences with molecular structures, which allows our model to learn more informative representations than baseline methods that use single modality data (i.e. Ryu et al. 2022). On the other hand, in the molecular graph structure encoder, we introduce graph contrastive learning to capture the dependencies between local and global structures, which enhances the representation learning of our model. We will demonstrate it in the following ablation study.\\n\\n3.4 Independent evaluation on the external dataset\\n\\nTo further validate the effectiveness and generalizability of our model, we carried out CMMS-GCL and baseline methods on an external dataset. Note that all the methods utilize the HLM dataset for model training and use the external data as a test dataset.\\n\\nThe results in Table 3 show that CMMS-GCL achieved the best performance compared with seven methods across all metrics on 10 times average results. More specifically, CMMS-GCL surpasses the second-best method by 6.60%, 8.17%, 4.45%, and 27.31% in terms of AUC, Accuracy, F1 score, and MCC, respectively. Furthermore, we derived a subset of molecules from the external dataset, which shows low similarity to the HLM dataset. We also compared our CMMS-GCL with various baselines on this subset of molecules with low structural similarity (<0.5) as shown in Table 4. We can observe the improvement of our method upon the second-best by 7.72%, 6.31%, 3.19%, and 38.60% in terms of AUC, Accuracy, F1 score, and MCC, respectively. More importantly, compared with the results on the HLM dataset (Table 2), CMMS-GCL exhibited a more substantial improvement on the external dataset (Tables 3 and 4). In contrast, for baseline methods, especially methods that are based on traditional machine learning, their performance decrease to varying degrees. Therefore, we can make a conclusion that the proposed CMMS-GCL is an effective and robust computational model for predicting the metabolic stability of compounds.\\n\\nPerformance evaluation of CMMS-GCL with baseline methods on the external datasets.\\n\\nThe best results are marked in bold and the second best is underlined.\\n\\nPerformance evaluation of CMMS-GCL against baseline methods using a subset of external datasets with low molecular similarity.\\n\\nThe best results are marked in bold and the second best is underlined.\\n\\n3.5 Ablation studies\\n\\nOur CMMS-GCL consists of an atom similarity-based sequence encoder, a molecular graph structure encoder, an inter-view graph contrastive learning, and a stability predictor, as shown in Fig. 1. In this section, we perform an ablation study on two datasets to evaluate the contributions of the cross-modality encoder and the inter-view graph contrastive learning. Note that in Tables 5 and 6, “w/o CM” denotes the variant of CMMS-GCL only reserves molecular graph structure encoder and stability predictor. “w/o GCL” denotes the variant of CMMS-GCL without inter-view graph contrastive learning. Tables 5 and 6 report the performance comparison between CMMS-GCL and its variants on HLM and the external dataset in terms of four metrics, respectively. It can be discovered that both cross-modality encoder and inter-view graph contrastive learning contribute to the performance improvement of the model. Specifically, in Table 5, compared with the variants w/o CM and w/o GCL, CMMS-GCL improves on AUC by 0.81% and 0.58%, on Accuracy by 1.13% and 0.75%, on F1 score by 1.55% and 0.95%, on MCC by 1.60% and 0.88%, respectively. Moreover, on the external dataset, compared with the variants w/o CM and w/o GCL, CMMS-GCL improves on AUC by 2.19% and 1.49%, on Accuracy by 2.46% and 1.46%, on F1 score by 1.48% and 0.56%, on MCC by 14.40% and 12.", "summary": "CMMS-GCL was developed to address the prediction task of metabolic stability of compound candidates. The model combines both compound SMILES sequences with molecular structures to learn more informative representations. It also introduces graph contrastive learning to capture dependencies between local and global structures, enhancing representation learning. The model was evaluated on the HLM dataset and an external dataset, achieving the best performance compared to baseline methods in terms of AUC, Accuracy, F1 score, and MCC.", "relevance": 10, "cited": true}]}, {"id": "joint-protocol-insufficient", "question": "What shared train, validation, and test split should be used for a joint benchmark of Open Babel and CMMS-GCL?", "answer": "I cannot answer. The context does not provide information about shared train, validation, and test splits for a joint benchmark of Open Babel and CMMS-GCL (openbabel-2011 chunk 6). The excerpt discusses validation and testing procedures of Open Babel, including an extensive test suite with 60 different test programs, nightly builds, and testing against specific validation suites such as the MMFF94 validation suite, as well as testing file format translation for SMILES and MDL Molfile formats. However, it does not mention any shared data splits for a joint benchmark with CMMS-GCL. Therefore, no specific train, validation, or test split is provided in the context for such a benchmark.", "references": "1. (openbabel-2011 chunk 6): O\'Boyle NM, Banck M, James CA, Morley C, Vandermeersch T, Hutchison GR. Open Babel: An open chemical toolbox. J Cheminform. 2011;3:33. doi:10.1186/1758-2946-3-33.", "elapsed_seconds": 12.584, "contexts": [{"document_id": "openbabel-2011", "chunk_name": "openbabel-2011 chunk 6", "source_text": " footprint of Open Babel, allowing external developers to choose particular functionality needed for their application and ignore other, less relevant features. It also allows the possibility of a third-party distributing plugins separately to the Open Babel distribution to provide additional functionality.\\n\\nOpen-Source License and Open Development\\n\\nOpen Babel is open-source software, which offers end users and third-party developers a range of additional rights not granted by proprietary chemistry software. Open-source software, at its most basic level, grants users the rights to study how their software works, to adapt it for any purpose or otherwise modify it, and to share the software and their modifications with others. In this sense, Open Source functions in similar ways to the processes of open peer review, publication, and citation in science. The rights granted by open source licenses largely coincide with the norms of scientific ethics to enable verifiability, repeatability, and building on previous results and theories.\\n\\nBeyond these rights, Open Babel (like most other open-source projects) offers open development -- that is, all development occurs in public forums and with public code repositories. This results in greater input from the community as any user can easily submit bug reports or feature suggestions, get involved in discussions on the future direction of Open Babel or even become a developer him/herself. In practice, the number of active contributors has increased over time through this level of open, public development (Figure 3). Moreover, it means that the development of the code is completely transparent and the quality of the software is available for public scrutiny. Indeed, since its inception, over 658 bugs have been submitted to the public tracker and fixed [41].\\n\\nNumber of contributors over time. Note that this graph only includes developers who directly commited code to the Open Babel source code repository, and does not include patches provided by users.\\n\\nValidation and Testing\\n\\nOpen Babel includes an extensive test suite comprising 60 different test programs each with tens to hundreds of tests. In early 2010, a nightly build infrastructure and dashboard was put in place with support from Kitware, Inc. This has greatly improved code quality by catching regressions, and also ensures that the code compiles cleanly on all platforms and compilers supported by Open Babel. Some examples of tests that are run each night are:\\n\\n(1) The MMFF94 forcefield code is tested against the MMFF94 validation suite.\\n\\n(2) The OBAlign class, which was developed using Test-Driven Development (TDD) methodology, is run against its test suite.\\n\\n(3) Handling of symmetry is validated by converting several test cases between SMILES, 2D and 3D SDF, and InChI (there are also several test programs with unit tests for the individual stereo classes in the API).\\n\\n(4) The SMARTS parser is tested using over 250 valid and invalid SMARTS patterns, and the SMARTS matcher is tested using 125 basic SMARTS patterns.\\n\\n(5) The LSSR (Least Set of Smallest Rings) code is tested for invariance against changing the atom order for a series of polycyclic molecules.\\n\\nRecently the development team has placed a major focus on increasing the robustness of file format translation particularly in relation to the commonly used SMILES and MDL Molfile formats. Translating between these formats requires accurate stereochemistry perception, inference of implicit hydrogens, and kekulization of delocalized systems. While it is difficult to ensure that any complex piece of code is free of bugs, and Open Babel is no exception, validation procedures can be carried out to assess the current level of performance and to find additional test cases that expose bugs. The following procedure was used to guide the rewriting of stereochemistry code in Open Babel, a project that began in early 2009. Starting with a dataset of 18,084 3D structures from PubChem3D as an SDF file, we compared the result of (a) conversion to SMILES, followed by conversion of that to Canonical SMILES to (b) conversion directly to Canonical SMILES. This procedure can be used to flush out errors in reading the original SDF file, reading/writing SMILES (either due to stereochemistry errors or kekulization problems), and is also a test (to some extent) of the canonicalization code. At the time of starting this work (March 2009), the error rate found was 1424 (8%); by Oct 2009, combined work on stereochemistry, kekulization and canonicalization had reduced this to 190 (~1%), and continued improvements have reduced the number of errors down to two (shown in Figure 4) for Open Babel 2.3.1 (~0.01%). The first failure is due to a kekulization error in a polycyclic aromatic molecule incorporating heteroatoms: (a) gave c1ccc2c(c1)c1[nH][nH]c3c4c1c(c2)ccc4cc1c3cccc1 while (b) gave c1ccc2c(c1)c1nnc3c4c1c(c2)ccc4cc1c3cccc1. This error led to confusion over whether or not the aromatic nitrogens have hydrogens attached (they do not). The second failure involves confusion over the canonical stereochemistry at a bridgehead carbon: (a) gave C1CN2[C@@H](C1)CCC2 while (b) gave C1CN2[C@H](", "summary": "The excerpt discusses the validation and testing procedures of Open Babel, including an extensive test suite with 60 different test programs and nightly builds to ensure code quality. It mentions testing against specific validation suites, such as the MMFF94 validation suite, and testing file format translation, particularly for SMILES and MDL Molfile formats. However, it does not provide information about shared train, validation, and test splits for a joint benchmark of Open Babel and CMMS-GCL.", "relevance": 2, "cited": true}]}], "base_profile": {"paperqa_version": "2026.8.12", "embedding_ndim": 768, "settings": {"llm": "ollama/qwen3:8b", "llm_config": {"model_list": [{"model_name": "ollama/qwen3:8b", "litellm_params": {"model": "ollama/qwen3:8b", "api_base": "http://127.0.0.1:11434"}}]}, "summary_llm": "ollama/qwen3:8b", "summary_llm_config": {"model_list": [{"model_name": "ollama/qwen3:8b", "litellm_params": {"model": "ollama/qwen3:8b", "api_base": "http://127.0.0.1:11434"}}]}, "embedding": "ollama/nomic-embed-text:v1.5", "embedding_config": {"model_list": [{"model_name": "ollama/nomic-embed-text:v1.5", "litellm_params": {"model": "ollama/nomic-embed-text:v1.5", "api_base": "http://127.0.0.1:11434"}}]}, "agent": {"agent_llm": "ollama/qwen3:8b", "agent_llm_config": {"model_list": [{"model_name": "ollama/qwen3:8b", "litellm_params": {"model": "ollama/qwen3:8b", "api_base": "http://127.0.0.1:11434"}}]}}, "parsing": {"use_doc_details": false, "multimodal": false, "enrichment_llm": "ollama/qwen3:8b", "enrichment_llm_config": {"model_list": [{"model_name": "ollama/qwen3:8b", "litellm_params": {"model": "ollama/qwen3:8b", "api_base": "http://127.0.0.1:11434"}}]}}, "answer": {"evidence_retrieval": true, "evidence_skip_summary": false, "get_evidence_if_no_contexts": true, "max_concurrent_requests": 1}}}, "effective_settings": {"llm": "ollama_chat/qwen3:8b", "llm_config": {"model_list": [{"model_name": "ollama_chat/qwen3:8b", "litellm_params": {"model": "ollama_chat/qwen3:8b", "api_base": "http://127.0.0.1:11434", "temperature": 0, "max_tokens": 1024, "think": false}}]}, "summary_llm": "ollama_chat/qwen3:8b", "summary_llm_config": {"model_list": [{"model_name": "ollama_chat/qwen3:8b", "litellm_params": {"model": "ollama_chat/qwen3:8b", "api_base": "http://127.0.0.1:11434", "temperature": 0, "max_tokens": 1024, "think": false}}]}, "embedding": "ollama/nomic-embed-text:v1.5", "embedding_config": {"model_list": [{"model_name": "ollama/nomic-embed-text:v1.5", "litellm_params": {"model": "ollama/nomic-embed-text:v1.5", "api_base": "http://127.0.0.1:11434"}}]}, "agent": {"agent_llm": "ollama_chat/qwen3:8b", "agent_llm_config": {"model_list": [{"model_name": "ollama_chat/qwen3:8b", "litellm_params": {"model": "ollama_chat/qwen3:8b", "api_base": "http://127.0.0.1:11434", "temperature": 0, "max_tokens": 1024, "think": false}}]}}, "parsing": {"use_doc_details": false, "multimodal": false, "enrichment_llm": "ollama_chat/qwen3:8b", "enrichment_llm_config": {"model_list": [{"model_name": "ollama_chat/qwen3:8b", "litellm_params": {"model": "ollama_chat/qwen3:8b", "api_base": "http://127.0.0.1:11434", "temperature": 0, "max_tokens": 1024, "think": false}}]}}, "answer": {"evidence_retrieval": true, "evidence_skip_summary": false, "get_evidence_if_no_contexts": true, "max_concurrent_requests": 1, "evidence_k": 4, "answer_max_sources": 3}}, "source_review": [{"question_id": "openbabel-format-coverage", "verdict": "core_supported", "source_document_id": "openbabel-2011", "source_quote": "Open Babel version 2.3 interconverts over 110 formats.", "limitation": "The answer is much longer than needed and includes generic evaluative language. Core support does not certify every additional sentence."}, {"question_id": "openbabel-utilities", "verdict": "core_supported_with_extra_prose_error", "source_document_id": "openbabel-2011", "source_quote": "the fingerprint format [13] and fastsearch format [14] are used for similarity and substructure searching", "limitation": "These are two distinct utility formats, so the core answer is supported. The later phrase about generating molecular summaries misattributes the report format\'s function to the two named search formats. The unedited answer preserves this defect."}, {"question_id": "cmms-modalities", "verdict": "core_supported", "source_document_id": "cmms-gcl-2023", "source_quote": "two modality data, i.e. SMILES sequence and molecule graph", "limitation": "Encoder details are unnecessary expansion. This question checks the two input modalities, not independent validation of the method."}, {"question_id": "cmms-task", "verdict": "core_supported_scope_caution", "source_document_id": "cmms-gcl-2023", "source_quote": "CMMS-GCL for predicting the metabolic stability of drug candidates", "limitation": "The extra best-performance statement refers only to results reported by the paper under its HLM/external-data protocols and compared methods. This demonstration did not reproduce or independently verify those model-performance results."}, {"question_id": "joint-protocol-insufficient", "verdict": "appropriate_abstention_limited_retrieval", "source_document_id": "openbabel-2011", "source_quote": null, "limitation": "The model declined to invent a joint split. Only one Open Babel chunk was returned and cited; this is not proof of absence across both complete papers or the wider literature. It is one abstention example, not a calibrated reliability estimate."}]}')

## 3. 先检查来源对象，再看答案

对象是已提供的文章，不是智能体新发现的论文。输入从 Europe PMC XML 提取摘要和正文纯文本，
不包括参考文献、图像和完整表格。Open Babel（2011）采用 CC BY 2.0，CMMS-GCL（2023）采用
CC BY 4.0。每份文本保留题名、作者、DOI 与许可。这不是 PDF 阅读能力测试。

库将文本切块，用 768 维本地嵌入模型表示候选片段，检索证据、生成摘要，再回答问题。
固定设置检索四个片段，答案最多引用三个来源。**片段不等于独立论文，也不等于独立样本。**
下方输出两份文档与第一个问题，构成完整起始示例。

In [ ]:
for document in observed["documents"]:
    print(document["title"])
    print(document["citation"])
    print(document["url"], document["license_url"])
    print(document["text"][:700], "\n")
print(observed["questions"][0]["question"])

## 4. 检查来源复制与引用对应

该函数检查每个保存片段是否确实出现在对应输入文章中，以及被引用片段是否出现在引用列表。
它**不能**证明答案可由该片段推出。复制一致性与语义支持是两种不同的检查，下一节会展示区别。

In [ ]:
def verify_source_copies(example):
    """Check copied text and citation resolution, not semantic entailment."""
    documents = {doc["id"]: doc["text"] for doc in example["documents"]}
    checked = 0
    for answer in example["answers"]:
        for context in answer["contexts"]:
            text = documents.get(context["document_id"], "")
            if not context["source_text"] or context["source_text"] not in text:
                raise ValueError("Source text is not present in the named supplied document")
            if context["cited"] and context["chunk_name"] not in answer["references"]:
                raise ValueError("Cited chunk is missing from the answer references")
            checked += 1
    return {"answers": len(example["answers"]), "source_chunks_checked": checked}

copy_check = verify_source_copies(observed)
print(copy_check)

## 5. 对照答案、模型摘要和原文

将 `question_index` 改为 0 到 4，再执行本 cell。第 0 题开头的“超过 110 种格式”直接受到
Open Babel 摘要支持。不要用模型摘要替代原文。下方显示完整答案；开头正确不代表后面的扩写都正确。

In [ ]:
question_index = 0
selected = observed["answers"][question_index]
print("QUESTION:", selected["question"])
print("ANSWER:", selected["answer"])
print("REFERENCES:", selected["references"])
for context in selected["contexts"]:
    if context["cited"]:
        print("\nCITED CHUNK:", context["chunk_name"])
        print("MODEL SUMMARY:", context["summary"])
        print("ORIGINAL TEXT:", context["source_text"])

## 6. 保留实际出现的不足

四个可回答问题的核心答案都有原文支持。但第 2 题先给出 fingerprint 与 fastsearch，随后
笼统地把生成分子摘要的功能也归给它们。原文把该功能归于 **report format**。保存的答案保留了
这个扩写错误，没有为了显得正确而改写。

CMMS-GCL 答案扩写中的“best performance”指该论文在其协议和比较范围内报告的结果，不是
本教程的新复现或通用排名。负控题得到了拒答，但实际只返回一个 Open Babel 片段。这体现了
合适的克制，不证明两篇全文或整个文献中都不存在相关信息。

这些说明来自助手对来源的复核，不是人类评分或基准分数。两篇公开论文上的五个问题不能衡量
泛化、校准或检索的因果收益；模型可能在预训练时见过文章。本次没有观察答案后再调整设置。

In [ ]:
for row, review in zip(observed["answers"], observed["source_review"], strict=True):
    print(row["id"], "—", review["verdict"])
    print("Returned documents:", sorted({c["document_id"] for c in row["contexts"]}))
    print(review["limitation"], "\n")
print("Observed runtime (not an accuracy table):", observed["observations"])

## 7. 独立证据练习

不生成新答案，指出哪个 utility format 生成分子结构摘要。在下方两个字符串填写答案与支持句，
再查看参考解释。说明为什么这能纠正第 2 个保存答案，又不改变其核心搜索格式答案。本练习不自动
评分推理质量。

<details><summary>参考解释</summary>
report format 生成分子结构摘要；fingerprint 和 fastsearch 是两个搜索 utility formats。
同一段落讨论了三者，所以即使引用该段，具体的功能对应也可能错误。
</details>

In [ ]:
student_answer = ""
student_supporting_sentence = ""
document = next(d for d in observed["documents"] if d["id"] == "openbabel-2011")
support_is_verbatim = bool(student_supporting_sentence) and student_supporting_sentence in document["text"]
print("A supplied supporting sentence appears verbatim:", support_is_verbatim)
print("This literal check does not grade the explanation.")

## 8. 证明这个有限的来源检查能发现错误

修改真实检索片段的**副本**。检查器必须拒绝虚构文本，原始示例保持不变。这只验证复制检查，
不能说明它能发现所有无依据答案。

In [ ]:
altered = copy.deepcopy(observed)
altered["answers"][0]["contexts"][0]["source_text"] = "An invented sentence outside the supplied articles."
invented_chunk_rejected = False
try:
    verify_source_copies(altered)
except ValueError as error:
    invented_chunk_rejected = True
    print("Expected rejection:", error)
assert invented_chunk_rejected

## 可选：新的本地推理，不是默认练习

使用单独准备的 Linux/Jupyter 环境，包含 Python 3.11+、`paper-qa[pypdf]==2026.8.12`、
Ollama 0.33.3、`qwen3:8b` 和 `nomic-embed-text:v1.5`。真实集成运行采用 Python 3.13.13
与 RTX A6000，采样显存峰值为 7.334 GB；它不是最低要求，也不保证其他机器的表现。使用共享
算力前先估算资源并取得许可。模型下载合计约 5.5 GB，不含运行软件；没有托管 API 回退。

开启本地 cell **之前**，先准备软件包、模型和 tokenizer 资源。下面的安装命令需要网络，
不会在 notebook 中自动执行。请使用隔离环境，不要直接安装到系统环境：

```sh
python -m pip install 'paper-qa[pypdf]==2026.8.12'
OLLAMA_NO_CLOUD=1 OLLAMA_HOST=127.0.0.1:11434 OLLAMA_CONTEXT_LENGTH=8192 OLLAMA_NUM_PARALLEL=1 ollama serve
# 在另一个终端中，等待服务就绪后：
ollama pull qwen3:8b
ollama pull nomic-embed-text:v1.5
python -c 'import tiktoken; [tiktoken.get_encoding(n) for n in ("cl100k_base", "o200k_base")]'
```

不要启动重复服务或停止不属于自己的服务。可参考 [Ollama 安装说明](https://docs.ollama.com/quickstart)
与[云功能关闭说明](https://docs.ollama.com/faq)。本 notebook 不会代你安装 Ollama、修改凭据、
选择 GPU、启动服务、下载权重或恢复失败的下载。

准备好后，再将 `ENABLE_LOCAL_INFERENCE = True`，重新执行模式 cell 及后续 cell。
下方显式 SDK 步骤沿用已运行集成的调用方式。

在同一个 cell 中设置 `EXECUTION_PROFILE`。配置只改变各项上限：

| 上限 | `as-observed` | `cpu-extended` |
| --- | --- | --- |
| 每次请求的模型线程数 | 服务默认 | 4 |
| SDK 重试次数 | 库默认 | 0 |
| 单次聊天调用时限 | 库默认 | 900 秒 |
| 嵌入配置时限 | 库默认 | 600 秒 |
| 单篇文档时限 | 150 秒 | 600 秒 |
| 单个问题时限 | 600 秒 | 2400 秒 |

`as-observed` 保持已观测的五问题批次所用上限，那次运行使用 GPU。首次 CPU-only 尝试沿用
这组上限，在为首篇文档做 embedding 时于 150 秒处停止，没有产生答案。其日志显示 20 个计算
线程被限制在四个逻辑 CPU 上，这支持线程超额分配是一个促成因素，但没有证明它是唯一原因。

`cpu-extended` 复现唯一一次通过验收的 CPU 执行副本的运行时预算。该副本在一台机器上于
**774.12 秒**内完成全部 12 个代码 cell，其中约 **708.61 秒**发生在问答阶段，已超过
`as-observed` 的 600 秒问题时限。它产生的那一条新答案核心事实有支持，但随后扩大了转换
测试的结论，并加入原文没有给出的可靠性保证。原有五条答案保持不变。

选择该配置复制的是那份预算，不是那次结果：你的机器、核数与内存并不相同；该副本同时改动
线程与时限，因此无法区分两者各自的作用；而且在那次运行中嵌入超时并未传递到 HTTP 调用，
真正的外层约束是单篇文档时限。两个配置都不是经过验证的默认 CPU 配方，任何配置都不改动
科学设置。双语离线版均已执行。

## 9. 检查并配置每一个本地模型角色

下方可见辅助函数检查有限的基础配置，并应用真实运行使用的聊天接口和设置。索引与问答都
显式接收相同的设置和嵌入对象，不使用默认托管模型。运行时配置在最后一步应用，即在断言
设置仍等于已观测设置之后；`apply_runtime_budget` 会去掉传输类上限后比较两份设置，只要
还有其他字段不同就抛错，因此配置无法悄悄改动模型、检索或回答设置。Python socket 检查
仅允许本机 Ollama 端口；它不是操作系统沙盒，不能审计单独运行的服务或原生库网络行为。
检查钩子会保留在 kernel 中，需要重启 kernel 才能移除。依赖或服务缺失应报错，不触发
其他服务回退。

In [ ]:
def router(model: str) -> dict:
    return {
        "model_list": [
            {
                "model_name": model,
                "litellm_params": {"model": model, "api_base": "http://127.0.0.1:11434"},
            }
        ]
    }

def validate_profile(profile: dict) -> None:
    """Reject missing roles, unknown keys, cloud routes and incompatible shapes."""
    if not isinstance(profile, dict) or not isinstance(profile.get("settings"), dict):
        raise ValueError("Expected a profile object containing settings.")
    chat = profile["settings"].get("llm")
    if chat not in ("ollama/qwen3:8b", "ollama/qwen3:14b"):
        raise ValueError("Only local qwen3:8b and qwen3:14b teaching profiles are supported.")
    embedding = "ollama/nomic-embed-text:v1.5"
    expected = {
        "paperqa_version": "2026.8.12",
        "embedding_ndim": 768,
        "settings": {
            "llm": chat,
            "llm_config": router(chat),
            "summary_llm": chat,
            "summary_llm_config": router(chat),
            "embedding": embedding,
            "embedding_config": router(embedding),
            "agent": {"agent_llm": chat, "agent_llm_config": router(chat)},
            "parsing": {
                "use_doc_details": False,
                "multimodal": False,
                "enrichment_llm": chat,
                "enrichment_llm_config": router(chat),
            },
            "answer": {
                "evidence_retrieval": True,
                "evidence_skip_summary": False,
                "get_evidence_if_no_contexts": True,
                "max_concurrent_requests": 1,
            },
        },
    }
    # JSON comparison distinguishes booleans from numbers as well as extra keys.
    if json.dumps(profile, sort_keys=True) != json.dumps(expected, sort_keys=True):
        raise ValueError(
            "Profile differs from the supported local-only shape. Check every model role, "
            "loopback endpoint, parser flag and dimension; overrides/fallbacks are not allowed."
        )

def smoke_settings(profile: dict) -> dict:
    validate_profile(profile)
    settings = copy.deepcopy(profile["settings"])
    # Use the documented Ollama chat transport for explicit thinking control.
    roles = [
        (settings, "llm"),
        (settings, "summary_llm"),
        (settings["agent"], "agent_llm"),
        (settings["parsing"], "enrichment_llm"),
    ]
    for section, key in roles:
        model = section[key].replace("ollama/", "ollama_chat/", 1)
        section[key] = model
        entry = section[key + "_config"]["model_list"][0]
        entry["model_name"] = model
        entry["litellm_params"].update(model=model, temperature=0, max_tokens=1024, think=False)
    settings["answer"].update(evidence_k=4, answer_max_sources=3)
    return settings

def scientific_settings(value):
    """Drop transport limits; what remains must be identical across profiles."""
    transport = {"num_thread", "timeout", "max_retries", "num_retries", "kwargs", "router_kwargs"}
    if isinstance(value, dict):
        return {k: scientific_settings(v) for k, v in value.items() if k not in transport}
    if isinstance(value, list):
        return [scientific_settings(item) for item in value]
    return value

def apply_runtime_budget(effective, budget):
    """Apply thread and timeout limits, then prove no scientific field moved."""
    updated = copy.deepcopy(effective)
    if budget["num_thread"] is not None:
        for section, key in [
            (updated, "llm"),
            (updated, "summary_llm"),
            (updated["agent"], "agent_llm"),
            (updated["parsing"], "enrichment_llm"),
        ]:
            section[key + "_config"]["model_list"][0]["litellm_params"].update(
                num_thread=budget["num_thread"],
                timeout=budget["chat_timeout"],
                max_retries=budget["max_retries"],
            )
        embedding = updated["embedding_config"]
        embedding["model_list"][0]["litellm_params"]["num_thread"] = budget["num_thread"]
        embedding["kwargs"] = {
            "timeout": budget["embedding_timeout"],
            "num_retries": budget["max_retries"],
        }
        embedding["router_kwargs"] = {
            "timeout": budget["embedding_timeout"],
            "num_retries": budget["max_retries"],
        }
    if scientific_settings(updated) != scientific_settings(effective):
        raise ValueError("A runtime profile must not change model, retrieval or answer settings")
    return updated

def install_network_guard(events: list[dict]) -> None:
    def audit(event, args):
        if event == "socket.getaddrinfo":
            address = args[:2]
        elif event == "socket.connect":
            address = args[1]
        elif event == "socket.sendto":
            address = args[-1]
        else:
            return
        allowed = (
            isinstance(address, tuple)
            and len(address) >= 2
            and address[0] in {"127.0.0.1", "localhost", "::1"}
            and str(address[1]) == "11434"
        )
        events.append({"event": event, "address": str(address), "allowed": allowed})
        if not allowed:
            raise PermissionError("External network operation refused during local inference.")

    sys.addaudithook(audit)

if ENABLE_LOCAL_INFERENCE:
    local_network_events = []
    try:
        if importlib.metadata.version("paper-qa") != "2026.8.12":
            raise RuntimeError("This lesson requires paper-qa 2026.8.12")
        effective = smoke_settings(copy.deepcopy(observed["base_profile"]))
        assert effective == observed["effective_settings"]
        effective = apply_runtime_budget(effective, runtime_budget(EXECUTION_PROFILE))
        os.environ.update(LITELLM_LOCAL_MODEL_COST_MAP="True", HF_HUB_OFFLINE="1",
                          NO_PROXY="127.0.0.1,localhost,::1", no_proxy="127.0.0.1,localhost,::1")
        install_network_guard(local_network_events)
        from lmi import LiteLLMEmbeddingModel
        from paperqa import Docs, Settings
        settings = Settings(**effective)
        embedding = LiteLLMEmbeddingModel(name=settings.embedding, ndim=768, config=settings.embedding_config)
        llm, summary_llm = settings.get_llm(), settings.get_summary_llm()
        local_docs = Docs()
        print("Runtime limits:", runtime_budget(EXECUTION_PROFILE))
        print("Local objects constructed; no documents indexed yet.")
    except BaseException as error:
        save_student_execution(OUTPUT_DIR, "configure", live_answers, local_network_events, error)
        raise
    else:
        save_student_execution(OUTPUT_DIR, "configure", live_answers, local_network_events)
else:
    print("Offline mode: no SDK import, server contact or model load.")

## 10. 为两份文本建立索引

这里显式展示索引步骤，而不是调用隐藏流程的命令行 runner。文本只写入新的学生目录。
显式提供准确元数据，避免外部元数据查询。每次调用使用所选配置的单篇文档时限：
`as-observed` 为 150 秒，`cpu-extended` 为 600 秒。

In [ ]:
if ENABLE_LOCAL_INFERENCE:
    try:
        for document in observed["documents"]:
            path = OUTPUT_DIR / (document["id"] + ".txt")
            with path.open("x", encoding="utf-8") as stream:
                stream.write(document["text"])
            added = await asyncio.wait_for(local_docs.aadd(
                path, citation=document["citation"], docname=document["id"],
                dockey=document["id"], title=document["title"], settings=settings,
                llm_model=llm, embedding_model=embedding,
            ), timeout=runtime_budget(EXECUTION_PROFILE)["document_timeout"])
            if added is None:
                raise RuntimeError("Document was not indexed: " + document["id"])
            print("Indexed:", document["id"])
    except BaseException as error:
        save_student_execution(OUTPUT_DIR, "index", live_answers, local_network_events, error)
        raise
    else:
        save_student_execution(OUTPUT_DIR, "index", live_answers, local_network_events)

## 11. 提问并检查返回证据

本 cell 只把 `local_question` 作为问题传入，不发送参考答案、复核结论或保存的模型答案。
使用所选配置的单个问题时限：`as-observed` 为 600 秒，`cpu-extended` 为 2400 秒。
修改问题产生的是新的学生查询，不是另一次作者基准实验。即使第一句正确，
也要检查完整回答及证据。温度为零仍不能保证新回答与保存答案完全相同。

In [ ]:
local_question = observed["questions"][0]["question"]
if ENABLE_LOCAL_INFERENCE:
    try:
        session = await asyncio.wait_for(local_docs.aquery(
            local_question, settings=settings, llm_model=llm,
            summary_llm_model=summary_llm, embedding_model=embedding,
        ), timeout=runtime_budget(EXECUTION_PROFILE)["question_timeout"])
        result = {
            "question": local_question, "answer": session.answer,
            "references": session.references,
            "contexts": [{"document_id": c.text.doc.dockey,
                          "chunk_name": c.text.name, "source_text": c.text.text,
                          "summary": c.context} for c in session.contexts],
        }
        live_answers.append(result)
        print(result["answer"])
        print(result["references"])
        display(result["contexts"])
        if any(not event["allowed"] for event in local_network_events):
            raise RuntimeError("External Python network attempt blocked; inspect student events")
    except BaseException as error:
        save_student_execution(OUTPUT_DIR, "answer", live_answers, local_network_events, error)
        raise
    else:
        save_student_execution(OUTPUT_DIR, "answer", live_answers, local_network_events)

## 12. 单独保存学生报告

报告记录运行模式与运行时配置、字面来源检查和练习回答，不把五个示例转换成准确率。作者的原始答案保持
不变。如需备份，请自行导出报告：保存 notebook 不等于备份输出目录。实时 cell 若失败，
应保留已有文件和错误，不要反复提问直到得到偏好的答案。

**来源。** [Open Babel](https://doi.org/10.1186/1758-2946-3-33)、
[CMMS-GCL](https://doi.org/10.1093/bioinformatics/btad503)、
[PaperQA 固定版本的 Docs 接口](https://github.com/Future-House/paper-qa/blob/v2026.08.12/src/paperqa/docs.py)。
第三方文章文本保留前述许可；原创教程文字采用 CC BY 4.0，原创教程代码采用 Apache-2.0。

In [ ]:
student_report = {
    "mode": "fresh_local_query" if ENABLE_LOCAL_INFERENCE else "read_saved_answers",
    "execution_profile": EXECUTION_PROFILE,
    "runtime_budget": runtime_budget(EXECUTION_PROFILE),
    "saved_answer_count": len(observed["answers"]),
    "source_copy_check": "passed",
    "invented_chunk_rejected": invented_chunk_rejected,
    "new_inference_performed": bool(live_answers),
    "exercise_answer": student_answer,
    "exercise_supporting_sentence": student_supporting_sentence,
    "support_is_verbatim": support_is_verbatim,
    "limitation": "Copy checks do not establish semantic support; no benchmark score is computed.",
}
path = OUTPUT_DIR / "student_report.json"
path.write_text(json.dumps(student_report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print(path)
display(student_report)